# A Practical Guide to Quantitative Finance Interviews — Chapter 6
## Finance / Option Pricing

Worked solutions to the option-pricing questions in **Xinfeng Zhou's Green Book**, in the style of the earlier
chapters: each concept is explained, then its interview questions are solved **step by step**, and every closed
form is cross-checked numerically. Concepts already built in Chapter 5 (Brownian motion, Itô's lemma) are
restated for reference where they are used.

### Notation used throughout Chapter 6
| symbol | meaning |
|---|---|
| $T$ | maturity date |
| $t$ | the current time |
| $\tau=T-t$ | time to maturity |
| $S$ | stock price at time $t$ |
| $r$ | continuous risk-free interest rate |
| $y$ | continuous dividend yield |
| $\sigma$ | annualized asset volatility |
| $c,\ p$ | price of a European **call** / **put** |
| $C,\ P$ | price of an American **call** / **put** |
| $D$ | present value, at $t$, of future dividends |
| $K$ | strike price |
| $PV$ | present value at $t$ |

(These are referenced by their full names now and then for ease of reading.)

## 6.1 Option Pricing

An **option** gives the right (not the obligation) to trade the stock at the strike $K$: a **call** to buy, a
**put** to sell. *European* options exercise only at maturity $T$; *American* options exercise any time up to
$T$. Everything below flows from **no-arbitrage** plus two Chapter-5 tools, restated here for reference:

- **Geometric Brownian motion (GBM)** for the stock: $dS=\mu S\,dt+\sigma S\,dW$, i.e.
  $S_T=S\,e^{(\mu-\sigma^2/2)\tau+\sigma(W_T-W_t)}$ — so $\ln S_T$ is normal (the stock is *log-normal*).
- **Itô's lemma:** for $f(t,S)$ with $dS=\mu S\,dt+\sigma S\,dW$,
$$df=\Big(f_t+\mu S f_S+\tfrac12\sigma^2S^2 f_{SS}\Big)dt+\sigma S f_S\,dW,$$
  and a process is a **martingale iff its $dt$ (drift) term is zero**.

The **risk-neutral** principle ties them together: under the risk-neutral measure the drift of the stock is the
risk-free rate ($\mu\to r$), and any European claim is worth the **discounted expected payoff**
$V=e^{-r\tau}\,E^{Q}[\text{payoff}(S_T)]$.

### 6.1.1 Price direction of options (1)

**Question.** How do vanilla European/American call and put prices move when each of $S,\ K,\ \tau,\ \sigma,\ r,\ D$
**increases** (holding the rest fixed)?

Reason variable by variable:
- **Stock price $S\uparrow$:** a call (right to *buy* at fixed $K$) gains; a put (right to *sell*) loses.
- **Strike $K\uparrow$:** buying becomes worse (call $\downarrow$), selling becomes better (put $\uparrow$).
- **Time to maturity $\tau\uparrow$:** an **American** option only gains from more time (more exercise
  opportunities), so both $C,P\uparrow$. A **European** option usually gains too, but not always — a deep
  in-the-money European *put* can *lose* value with more time (its payoff is capped at $K$, whose present value
  shrinks), so European $\tau$ is **ambiguous**.
- **Volatility $\sigma\uparrow$:** options are convex in $S$ (limited downside, open upside), so more dispersion
  helps **both** calls and puts ($\uparrow$).
- **Interest rate $r\uparrow$:** raising $r$ discounts the strike more. A call (pays $-K$ later) gains; a put
  (receives $+K$ later) loses.
- **Dividends $D\uparrow$:** dividends drop the stock on the ex-date, hurting calls and helping puts.

**Summary — effect of *increasing* each variable:**

| increase ↑ | European call $c$ | European put $p$ | American call $C$ | American put $P$ |
|---|:---:|:---:|:---:|:---:|
| $S$ (stock) | $+$ | $-$ | $+$ | $-$ |
| $K$ (strike) | $-$ | $+$ | $-$ | $+$ |
| $\tau$ (time to maturity) | ? | ? | $+$ | $+$ |
| $\sigma$ (volatility) | $+$ | $+$ | $+$ | $+$ |
| $r$ (rate) | $+$ | $-$ | $+$ | $-$ |
| $D$ (dividends) | $-$ | $+$ | $-$ | $+$ |

("$?$" marks the genuinely ambiguous European-time-value cases; American values are monotone in $\tau$ because
a longer-dated American option contains the shorter-dated one as an exercise choice.) The code confirms every
unambiguous sign by bumping each input in the Black–Scholes price.

In [1]:
import math

def N(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def bs(S, K, r, sigma, T, call=True, q=0.0):
    """Black-Scholes price with continuous dividend yield q (=y)."""
    d1 = (math.log(S / K) + (r - q + sigma ** 2 / 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if call:
        return S * math.exp(-q * T) * N(d1) - K * math.exp(-r * T) * N(d2)
    return K * math.exp(-r * T) * N(-d2) - S * math.exp(-q * T) * N(-d1)

base = dict(S=100, K=100, r=0.05, sigma=0.2, T=1.0, q=0.02)   # q = dividend yield D
sign = lambda x: "+" if x > 1e-9 else ("-" if x < -1e-9 else "0")
def price(call, **over):
    a = dict(base); a.update(over)
    return bs(a['S'], a['K'], a['r'], a['sigma'], a['T'], call=call, q=a['q'])

c0, p0 = price(True), price(False)
print("effect of increasing each variable on the Black-Scholes price (sign of the change):")
print(f"{'var':>6} | call | put")
for var, over in [('S', dict(S=101)), ('K', dict(K=101)), ('tau', dict(T=1.01)),
                  ('sigma', dict(sigma=0.21)), ('r', dict(r=0.06)), ('D(q)', dict(q=0.03))]:
    print(f"{var:>6} |  {sign(price(True, **over)-c0)}   |  {sign(price(False, **over)-p0)}")

effect of increasing each variable on the Black-Scholes price (sign of the change):
   var | call | put
     S |  +   |  -
     K |  -   |  +
   tau |  +   |  +
 sigma |  +   |  +
     r |  +   |  -
  D(q) |  -   |  +


### 6.1.2 Put-call parity (2)

**Question.** State and prove put–call parity for **European** options on a **non-dividend** stock.

**Statement.**
$$\boxed{\,c+K e^{-r\tau}=p+S\,}.$$

**Proof by two portfolios with identical maturity payoffs.** Build:
- **Portfolio A:** one European **call** ($c$) $+$ a **zero-coupon bond** with face value $K$ (present value
  $Ke^{-r\tau}$). Cost today: $c+Ke^{-r\tau}$.
- **Portfolio B:** one European **put** ($p$) $+$ one **share** of the stock ($S$) — a *protective put*. Cost
  today: $p+S$.

Their payoffs at maturity $T$:

| at $T$ | $S_T\le K$ | $S_T>K$ |
|---|:---:|:---:|
| **A:** call $+$ bond | $0+K=K$ | $(S_T-K)+K=S_T$ |
| **B:** put $+$ stock | $(K-S_T)+S_T=K$ | $0+S_T=S_T$ |

Both portfolios pay $\max(S_T,K)$ in *every* state. Two portfolios with identical payoffs in all states must
have the same price today (else short the dearer, buy the cheaper for a riskless profit), so
$c+Ke^{-r\tau}=p+S$. The code checks it against Black–Scholes.

In [2]:
# Put-call parity check: c + K e^{-r tau} should equal p + S  (non-dividend: q = 0)
S, K, r, sigma, T = 100, 95, 0.05, 0.2, 1.0
c = bs(S, K, r, sigma, T, call=True, q=0.0)
p = bs(S, K, r, sigma, T, call=False, q=0.0)
lhs = c + K * math.exp(-r * T)      # portfolio A: call + zero-coupon bond (face K)
rhs = p + S                          # portfolio B: put + stock (protective put)
print(f"c = {c:.4f},  p = {p:.4f}")
print(f"A: c + K e^-rT = {lhs:.6f}")
print(f"B: p + S       = {rhs:.6f}   -> equal, parity holds")

c = 13.3465,  p = 3.7133
A: c + K e^-rT = 103.713260
B: p + S       = 103.713260   -> equal, parity holds


### 6.1.3 American vs European options (3)

#### 3A — never exercise an American call early on a non-dividend stock

With no dividends the American and European call have the **same** price, because it is never optimal to
exercise the American call before maturity. Three ways to see it.

**Solution 1 — the lower bound beats the exercise value.** For a non-dividend European call,
$$c\ \ge\ S-Ke^{-r\tau}\ >\ S-K\qquad(\text{since } Ke^{-r\tau}<K\text{ for } r>0).$$
The call *alive* is worth more than $S-K$, which is exactly what you would collect by exercising. Exercising
throws away both the remaining time value and the interest you could earn on the strike, so you never do it —
$C=c$.

**Solution 2 — dominate the early-exercise cash flows.** Suppose at time $t\,(<T)$ the call is in the money and
you are tempted to exercise. Compare two strategies:

| | at time $t$ | at maturity $T$ |
|---|---|---|
| **1. Exercise now**, sell the share, invest the cash | net $S_t-K$, invested at $r$ | $(S_t-K)e^{r\tau}$ |
| **2. Keep the call**, short one share, invest $S_t$ | short $+S_t$, invest it | $S_t e^{r\tau}-\min(S_T,K)$ |

In strategy 2 you close the short at $T$ by buying the share in the market for $S_T$ **or** exercising the call
for $K$, whichever is cheaper — cost $\min(S_T,K)$. The difference is
$$\text{(2)}-\text{(1)}=Ke^{r\tau}-\min(S_T,K)\ \ge\ Ke^{r\tau}-K=K\big(e^{r\tau}-1\big)>0.$$
Keeping the call **always** beats exercising, by at least $K(e^{r\tau}-1)$. So never exercise early.

**Solution 3 — Jensen's inequality (convex payoff).** The call payoff $(S-K)^+$ is a **convex** function of
$S$. Under the risk-neutral measure $E^Q[S_T]=Se^{r\tau}$, so by Jensen,
$$c=e^{-r\tau}E^Q\big[(S_T-K)^+\big]\ \ge\ e^{-r\tau}\big(E^Q[S_T]-K\big)^+=e^{-r\tau}\big(Se^{r\tau}-K\big)^+=\big(S-Ke^{-r\tau}\big)^+\ >\ S-K.$$
Again the alive call exceeds the exercise value $S-K$. The convexity is the picture below: the chord of a
convex payoff lies below the curve, so averaging *then* taking the payoff (holding) beats taking the payoff of
the average (exercising now).
```
  payoff
    |                         /(S-K)^+
    |                        /
    |          hold: E[(S_T-K)^+]  •   (on the curve, above the chord)
    |                      /  ·
    |                     / ·   chord (Jensen gap)
    |____________________/·___________________  S
    0                    K        Se^{r tau}
```

(With dividends the story changes — a large enough dividend can make early exercise of an American call
optimal just before the ex-date; that is the one exception.)

#### 3B — is there an arbitrage between a call struck at 80 and a put struck at 90?

**Question.** A European **call** on a non-dividend stock with strike $80$ trades at $8$; a European **put** on
the same stock with strike $90$ trades at $9$. Is there an arbitrage?

**Set up a candidate arbitrage and check its payoff.** Try the portfolio that is long the (cheaper, lower-strike)
call and short the (dearer, higher-strike) put, delta-hedged with the stock and a bond:
$$\text{long call}(80),\quad \text{short put}(90),\quad \text{short 1 share},\quad \text{lend } 90e^{-r\tau}.$$
Its payoff at $T$ is
$$(S_T-80)^+-(90-S_T)^+-S_T+90=\begin{cases}0,& S_T<80,\\ S_T-80\in[0,10),& 80\le S_T<90,\\ 10,& S_T\ge90,\end{cases}$$
which is $\ge0$ always and $>0$ whenever $S_T>80$ (positive probability). So the **payoff leg is a valid
arbitrage payoff** — the only question is whether it costs nothing to set up. Its cost today is
$$8-9-S+90e^{-r\tau}=90e^{-r\tau}-S-1.$$
So this is a genuine arbitrage **iff** $S\ge 90e^{-r\tau}-1$. The mirror portfolio (long put(90), short call(80),
long stock, borrow $80e^{-r\tau}$) is an arbitrage when $S\le 80e^{-r\tau}-1$. In between —
$$80e^{-r\tau}-1\ <\ S\ <\ 90e^{-r\tau}-1$$
— **neither works, and the two prices are arbitrage-free.**

**So the honest answer: it depends on the current stock price.** The two option quotes *alone* do not force an
arbitrage. For example at $S=85,\ r=0$ they are perfectly consistent: put–call parity at strike $90$ gives the
implied call $c(90)=p(90)+S-90=9+85-90=4$, and $c(80)-c(90)=8-4=4$ lies in the required $[0,\,90-80]=[0,10]$, so
a valid price system (a risk-neutral distribution) exists. But at $S=95,\ r=0$ the first portfolio costs
$90-95-1=-6<0$ — a free $6$ today with a non-negative future payoff — a clear arbitrage. The code prints the
setup cost across stock prices and the no-arbitrage band.

In [3]:
import math

def arb_costs(S, r=0.0, T=1.0):
    """Setup cost of the two candidate arbitrage portfolios for call(80)=8, put(90)=9.
    Negative cost with a >=0 payoff = arbitrage."""
    high = 8 - 9 - S + 90 * math.exp(-r * T)      # long call80, short put90, short stock, lend 90 e^-rT
    low = 9 - 8 + S - 80 * math.exp(-r * T)       # mirror
    return high, low

def high_payoff(ST):
    return max(ST - 80, 0) - max(90 - ST, 0) - ST + 90

print("payoff of the long-call/short-put/short-stock/lend portfolio (>=0 always):")
print("  S_T = 50, 85, 120  ->", [high_payoff(s) for s in (50, 85, 120)])
print("\nsetup cost by current stock price (r = 0):  arbitrage when a cost is negative")
for S in (70, 79, 85, 89, 95):
    hi, lo = arb_costs(S)
    tag = "ARB" if (hi < -1e-9 or lo < -1e-9) else "no-arb"
    print(f"  S={S:>3}: high-cost {hi:+.1f}, low-cost {lo:+.1f}   -> {tag}")
print("\nno-arbitrage band (r=0): 79 < S < 89  (there the quotes are consistent, e.g. S=85)")

payoff of the long-call/short-put/short-stock/lend portfolio (>=0 always):
  S_T = 50, 85, 120  -> [0, 5, 10]

setup cost by current stock price (r = 0):  arbitrage when a cost is negative
  S= 70: high-cost +19.0, low-cost -9.0   -> ARB
  S= 79: high-cost +10.0, low-cost +0.0   -> no-arb
  S= 85: high-cost +4.0, low-cost +6.0   -> no-arb
  S= 89: high-cost +0.0, low-cost +10.0   -> no-arb
  S= 95: high-cost -6.0, low-cost +16.0   -> ARB

no-arbitrage band (r=0): 79 < S < 89  (there the quotes are consistent, e.g. S=85)


### 6.1.4 The Black–Scholes–Merton differential equation (4)

**Question.** Write down the BSM PDE and sketch its derivation.

**The equation.** For a derivative $V(S,t)$ on a non-dividend stock,
$$\boxed{\ \frac{\partial V}{\partial t}+\tfrac12\sigma^{2}S^{2}\frac{\partial^{2}V}{\partial S^{2}}+rS\frac{\partial V}{\partial S}-rV=0\ }.$$

**Derivation (GBM + Itô + a riskless hedge).**
1. **Model the stock** as GBM: $dS=\mu S\,dt+\sigma S\,dW$.
2. **Itô's lemma** on $V(S,t)$:
$$dV=\Big(V_t+\mu S V_S+\tfrac12\sigma^{2}S^{2}V_{SS}\Big)dt+\sigma S V_S\,dW.$$
3. **Form a hedged portfolio** that cancels the random $dW$: hold the derivative and short $\Delta=V_S$ shares,
$\Pi=V-V_S\,S$. Then
$$d\Pi=dV-V_S\,dS=\Big(V_t+\tfrac12\sigma^{2}S^{2}V_{SS}\Big)dt,$$
the $\mu S V_S\,dt$ and $\sigma S V_S\,dW$ terms **cancel** — $\Pi$ is instantaneously **riskless**.
4. **No arbitrage** forces a riskless portfolio to earn $r$: $d\Pi=r\Pi\,dt=r\big(V-V_S S\big)dt$. Equating the
   two expressions for $d\Pi$ and cancelling $dt$,
$$V_t+\tfrac12\sigma^{2}S^{2}V_{SS}=r\big(V-S V_S\big)\ \Longrightarrow\ V_t+\tfrac12\sigma^{2}S^{2}V_{SS}+rSV_S-rV=0.$$

Notice $\mu$ has **vanished**: the hedge removes the drift, which is why option prices do not depend on the
stock's expected return — the essence of **risk-neutral** pricing. Each option is this same PDE with a
different terminal condition (a call: $V(S,T)=(S-K)^+$; a put: $(K-S)^+$).

### 6.1.5 The Black–Scholes formula (5)

With continuous dividend yield $y$, the European call and put are
$$c=Se^{-y\tau}N(d_1)-Ke^{-r\tau}N(d_2),\qquad p=Ke^{-r\tau}N(-d_2)-Se^{-y\tau}N(-d_1),$$
$$d_1=\frac{\ln(Se^{-y\tau}/K)+(r+\sigma^2/2)\tau}{\sigma\sqrt\tau}=\frac{\ln(S/K)+(r-y+\sigma^2/2)\tau}{\sigma\sqrt\tau},\qquad
d_2=\frac{\ln(S/K)+(r-y-\sigma^2/2)\tau}{\sigma\sqrt\tau}=d_1-\sigma\sqrt\tau,$$
where $N(x)=\int_{-\infty}^{x}\tfrac{1}{\sqrt{2\pi}}e^{-y^2/2}\,dy$ is the standard normal CDF and
$N'(x)=\tfrac{1}{\sqrt{2\pi}}e^{-x^2/2}$ its density. (For a futures underlying $y=r$; for an FX rate $y=r_f$,
the foreign rate.) The derivations below take the non-dividend case $y=0$.

#### 5A — the six assumptions behind Black–Scholes
1. The stock follows a **geometric Brownian motion** with **constant** drift $\mu$ and volatility $\sigma$
   (so $S_T$ is log-normal).
2. **No arbitrage.**
3. **Frictionless, continuous trading:** securities are infinitely divisible, short selling is allowed, and you
   can rehedge continuously.
4. **No transaction costs or taxes.**
5. A **constant** risk-free rate $r$ at which you may borrow and lend freely.
6. **No dividends** over the life of the option (the base version; a yield $y$ is the standard extension).

#### 5B — deriving the call by risk-neutral expectation

Under the **risk-neutral measure** the stock drifts at $r$:
$$S_T=S\,e^{(r-\sigma^2/2)\tau+\sigma\sqrt\tau\,Z},\qquad Z\sim N(0,1),$$
and the call is the discounted expected payoff $c=e^{-r\tau}E^{Q}\big[(S_T-K)^+\big]$.

**Step 1 — when is the call in the money?** $S_T>K\iff Z>\dfrac{\ln(K/S)-(r-\sigma^2/2)\tau}{\sigma\sqrt\tau}=:-d_2$.
So the option pays off for $Z>-d_2$.

**Step 2 — split the expectation.**
$$e^{r\tau}c=E^{Q}\big[(S_T-K)^+\big]=\underbrace{E^{Q}\big[S_T\,\mathbf 1_{Z>-d_2}\big]}_{(\mathrm I)}-\underbrace{K\,E^{Q}\big[\mathbf 1_{Z>-d_2}\big]}_{(\mathrm{II})}.$$

**Step 3 — the easy term (II).** $E^{Q}[\mathbf 1_{Z>-d_2}]=P(Z>-d_2)=N(d_2)$, so $(\mathrm{II})=K\,N(d_2)$.

**Step 4 — the stock term (I), completing the square.** With $\varphi(z)=\tfrac1{\sqrt{2\pi}}e^{-z^2/2}$,
$$(\mathrm I)=\int_{-d_2}^{\infty}S\,e^{(r-\sigma^2/2)\tau+\sigma\sqrt\tau\,z}\,\varphi(z)\,dz
=Se^{r\tau}\int_{-d_2}^{\infty}\tfrac1{\sqrt{2\pi}}e^{-\frac12(z-\sigma\sqrt\tau)^2}\,dz,$$
because $(r-\sigma^2/2)\tau+\sigma\sqrt\tau z-\tfrac{z^2}{2}=r\tau-\tfrac12(z-\sigma\sqrt\tau)^2$. Substituting
$w=z-\sigma\sqrt\tau$ turns the integral into $P(w>-d_2-\sigma\sqrt\tau)=N(d_2+\sigma\sqrt\tau)=N(d_1)$, so
$(\mathrm I)=Se^{r\tau}N(d_1)$.

**Step 5 — assemble and discount.**
$$e^{r\tau}c=Se^{r\tau}N(d_1)-K\,N(d_2)\ \Longrightarrow\ \boxed{\,c=S\,N(d_1)-Ke^{-r\tau}N(d_2)\,}.$$
The put follows from put–call parity. The code prices both and re-checks parity.

In [4]:
import math, random

def bs_call(S, K, r, sigma, T):
    d1 = (math.log(S / K) + (r + sigma ** 2 / 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    return S * N(d1) - K * math.exp(-r * T) * N(d2)

def call_montecarlo(S, K, r, sigma, T, trials=1000000, seed=0):
    """c = e^{-rT} E^Q[(S_T - K)^+] under risk-neutral GBM."""
    rng = random.Random(seed); tot = 0.0
    for _ in range(trials):
        ST = S * math.exp((r - sigma ** 2 / 2) * T + sigma * math.sqrt(T) * rng.gauss(0, 1))
        tot += max(ST - K, 0.0)
    return math.exp(-r * T) * tot / trials

S, K, r, sigma, T = 100, 100, 0.05, 0.2, 1.0
print(f"call (formula)      = {bs_call(S,K,r,sigma,T):.4f}")
print(f"call (risk-neutral MC) = {call_montecarlo(S,K,r,sigma,T):.4f}")
p = bs(S, K, r, sigma, T, call=False, q=0.0)
print(f"put via parity check: c - p = {bs_call(S,K,r,sigma,T)-p:+.4f}  vs  S - K e^-rT = {S-K*math.exp(-r*T):+.4f}")

call (formula)      = 10.4506
call (risk-neutral MC) = 10.4645
put via parity check: c - p = +4.8771  vs  S - K e^-rT = +4.8771


#### 5C — deriving the call by solving the BSM PDE (reduction to the heat equation)

Solve $V_t+\tfrac12\sigma^2S^2V_{SS}+rSV_S-rV=0$ with $V(S,T)=(S-K)^+$ by turning it into the **heat
equation** $u_\theta=u_{xx}$, whose solution is known.

**Step 1 — log-price and reversed, rescaled time.** Put
$$x=\ln\frac{S}{K},\qquad \theta=\tfrac12\sigma^2(T-t),\qquad V(S,t)=K\,v(x,\theta).$$
Then $\partial_t=-\tfrac12\sigma^2\partial_\theta$, $S\partial_S=\partial_x$, $S^2\partial_{SS}=\partial_{xx}-\partial_x$, and
dividing by $\tfrac12\sigma^2 K$ the PDE becomes, with $k=\dfrac{r}{\tfrac12\sigma^2}=\dfrac{2r}{\sigma^2}$,
$$v_\theta=v_{xx}+(k-1)v_x-k\,v.$$

**Step 2 — kill the lower-order terms.** Look for $v=e^{\alpha x+\beta\theta}u(x,\theta)$. Choosing
$$\alpha=-\tfrac{k-1}{2},\qquad \beta=-\tfrac{(k+1)^2}{4}$$
cancels the $v_x$ and $v$ terms and leaves the pure **heat equation**
$$u_\theta=u_{xx}.$$

**Step 3 — the initial condition.** At $\theta=0$ (i.e. $t=T$), $V=(S-K)^+=K(e^x-1)^+$, so $v(x,0)=(e^x-1)^+$
and hence
$$u(x,0)=e^{-\alpha x}(e^x-1)^+=\big(e^{\frac{k+1}{2}x}-e^{\frac{k-1}{2}x}\big)^+ .$$

**Step 4 — solve the heat equation** with its Gaussian kernel,
$$u(x,\theta)=\frac{1}{\sqrt{4\pi\theta}}\int_{-\infty}^{\infty}u(s,0)\,e^{-(x-s)^2/4\theta}\,ds.$$
Each of the two exponential pieces is a Gaussian integral that **completes the square** into a standard-normal
tail — exactly the two integrals of Step 4 in 5B — producing two $N(\cdot)$ terms.

**Step 5 — unwind the substitutions.** Reversing $u\to v\to V$ and rewriting $x,\theta,k$ back in terms of
$S,\tau,r,\sigma$, the two normal terms become $N(d_1)$ and $N(d_2)$ with the $d_1,d_2$ above, giving
$$\boxed{\,c=S\,N(d_1)-Ke^{-r\tau}N(d_2)\,}$$
— the same formula as the probabilistic route, now as the unique solution of the PDE with the call's boundary
condition.

#### 5D — an option that pays 1 the first time the stock hits $H$

**Question.** Interest rate is $0$; a non-dividend stock is at $1$ today. The first time the price reaches a
level $H>1$ you may exercise and receive $1$. What is the option worth today?

**Brute force (risk-neutral).** With $r=0$ and no dividends, the discounted price is a **martingale** and there
is no discounting, so the option is worth $P(\text{the stock ever reaches }H)\times 1$. The stock $S_t$ is a
positive martingale started at $1$; run it until it either hits $H$ or drifts arbitrarily low. Optional
stopping on the martingale $S$ gives
$$E[S_\tau]=S_0=1\ \Longrightarrow\ P(\text{hit }H)\cdot H+P(\text{never})\cdot 0=1\ \Longrightarrow\ P(\text{hit }H)=\frac1H.$$
So the option is worth $\dfrac1H$.

**No-arbitrage replication.** Buy $\tfrac1H$ shares today, at cost $\tfrac1H\cdot 1=\tfrac1H$. If the stock ever
reaches $H$, those shares are worth $\tfrac1H\cdot H=1$ — exactly the payoff, which you deliver by selling them.
If the stock never reaches $H$ (it drifts to $0$), both the shares and the option end at $0$. The stock position
reproduces the option's payoff in every scenario, so by no-arbitrage the option is worth
$$\boxed{\dfrac1H}.$$

In [5]:
import math, random

def one_touch_value(H):
    """Zero-rate, non-dividend: value of receiving 1 when S first hits H, starting at S0=1.  = 1/H."""
    return 1.0 / H

def one_touch_sim(H, sigma=0.5, dt=0.05, trials=8000, cap=150.0, floor=0.02, seed=0):
    """P(martingale stock started at 1 reaches H before drifting to ~0)."""
    rng = random.Random(seed); hit = 0
    for _ in range(trials):
        s = 1.0; t = 0.0
        while t < cap and floor < s < H:
            s *= math.exp(-0.5 * sigma * sigma * dt + sigma * math.sqrt(dt) * rng.gauss(0, 1)); t += dt
        hit += (s >= H)
    return hit / trials

for H in (2.0, 3.0, 4.0):
    print(f"H={H}: value 1/H = {one_touch_value(H):.4f}   (Monte-Carlo P(hit H) ~ {one_touch_sim(H):.3f})")

H=2.0: value 1/H = 0.5000   (Monte-Carlo P(hit H) ~ 0.470)
H=3.0: value 1/H = 0.3333   (Monte-Carlo P(hit H) ~ 0.315)
H=4.0: value 1/H = 0.2500   (Monte-Carlo P(hit H) ~ 0.239)


#### 5E — a contract that pays $1/S_T$ at maturity

**Question.** A non-dividend stock follows geometric Brownian motion. What is the value today of a contract
paying $1/S_T$ at maturity $T$?

**Solution — Itô under the risk-neutral measure.** Risk-neutrally $dS=rS\,dt+\sigma S\,dW$. Apply Itô's lemma to
$f(S)=1/S$ (here $f_S=-1/S^2$, $f_{SS}=2/S^3$):
$$d\!\left(\frac1S\right)=-\frac{1}{S^2}\,dS+\tfrac12\frac{2}{S^3}(dS)^2=\frac1S\Big[(\sigma^2-r)\,dt-\sigma\,dW\Big].$$
So $1/S$ is itself a geometric Brownian motion with drift $\sigma^2-r$, giving
$$E^{Q}\!\Big[\frac{1}{S_T}\Big]=\frac{1}{S}\,e^{(\sigma^2-r)\tau}.$$
Discounting at $r$,
$$V=e^{-r\tau}E^{Q}\!\Big[\frac{1}{S_T}\Big]=\boxed{\ \frac{1}{S}\,e^{(\sigma^2-2r)\tau}\ }.$$
(Equivalently, write $1/S_T=\tfrac1S e^{-(r-\sigma^2/2)\tau-\sigma\sqrt\tau Z}$ and use
$E[e^{-\sigma\sqrt\tau Z}]=e^{\sigma^2\tau/2}$ — the same answer.) The extra $e^{\sigma^2\tau}$ over naive
$\tfrac1S e^{-r\tau}$ is the convexity of $1/x$: by Jensen $E[1/S_T]>1/E[S_T]$.

In [6]:
import math, random

def inverse_contract_value(S, r, sigma, T):
    """Value of a claim paying 1/S_T:  (1/S) e^{(sigma^2 - 2r) T}."""
    return (1.0 / S) * math.exp((sigma ** 2 - 2 * r) * T)

def inverse_contract_sim(S, r, sigma, T, trials=800000, seed=0):
    rng = random.Random(seed); tot = 0.0
    for _ in range(trials):
        ST = S * math.exp((r - sigma ** 2 / 2) * T + sigma * math.sqrt(T) * rng.gauss(0, 1))
        tot += 1.0 / ST
    return math.exp(-r * T) * tot / trials

S, r, sigma, T = 100, 0.05, 0.3, 1.0
print(f"value of 1/S_T contract: formula {inverse_contract_value(S,r,sigma,T):.6f}   "
      f"(risk-neutral MC {inverse_contract_sim(S,r,sigma,T):.6f})")
print(f"naive (ignoring convexity) 1/S e^-rT = {(1/S)*math.exp(-r*T):.6f}  <  correct value (Jensen)")

value of 1/S_T contract: formula 0.009900   (risk-neutral MC 0.009896)
naive (ignoring convexity) 1/S e^-rT = 0.009512  <  correct value (Jensen)


## 6.2 The Greeks

The **Greeks** measure how an option's price $V$ reacts when *one* input moves a little. Every Greek is a
**partial derivative of the option price** with respect to one input — a **first-order** partial derivative for
Delta, Theta, Vega and Rho, and a **second-order** partial derivative for Gamma — so together they are the
coefficients of the price's Taylor expansion:
$$dV\;\approx\;\Delta\,dS+\tfrac12\,\Gamma\,(dS)^{2}+\Theta\,dt+\text{Vega}\,d\sigma+\rho\,dr .$$
(The notation of §6.1 carries over: stock price $S$, strike $K$, time to maturity $\tau=T-t$, volatility
$\sigma$, risk-free rate $r$, dividend yield $y$, European call and put prices $c$ and $p$.)

| Greek | definition | order | question it answers | everyday analogy |
|---|---|---|---|---|
| **Delta** $\Delta$ | $\partial V/\partial S$ | 1st | how many dollars does the option gain per 1 dollar move in the stock? | the **speedometer**: how fast the option moves relative to the stock |
| **Gamma** $\Gamma$ | $\partial^{2}V/\partial S^{2}=\partial\Delta/\partial S$ | 2nd | how fast does delta itself change as the stock moves? | the **accelerator**: how quickly the speedometer reading changes (twitchy steering) |
| **Theta** $\Theta$ | $\partial V/\partial t=-\partial V/\partial\tau$ | 1st | how much value does the option lose just from a day passing? | a **melting ice cube**: a perishable good that shrinks as time ticks by |
| **Vega** | $\partial V/\partial\sigma$ | 1st | how much does the price change per extra point of volatility? | a **storm barometer**: options are insurance, and choppier weather makes insurance dearer |
| **Rho** $\rho$ | $\partial V/\partial r$ | 1st | how much does the price change when interest rates move? | the **interest-rate dial**: paying (or receiving) the strike later is worth more or less as rates change |

**Reading the analogies.** Think of an option as an *insurance policy* on the stock. *Delta* says how much the
policy's price moves with the stock (the speedometer). *Gamma* says how quickly that speed changes
(acceleration — a high-gamma policy is twitchy). *Theta* is the policy's expiry clock: every day it gets closer to
running out, so it is worth a little less. *Vega* is how much more the insurer charges when the weather
(uncertainty) gets stormier. *Rho* is the small effect of interest rates, like the way a mortgage rate changes
the real cost of paying later.

| Greek | main use in practice | what drives it | typical sign |
|---|---|---|---|
| **Delta** | hedge ratio (shares to hold against one option); rough chance of finishing in the money; directional exposure | moneyness $S/K$ (most), time to maturity, volatility; pushed towards 0 or 1 as $\tau\to0$ | call: between 0 and 1; put: between -1 and 0 |
| **Gamma** | stability of the delta hedge — how often you must rebalance; the option's convexity | largest at the money, explodes as $\tau\to0$, falls as $\sigma$ rises | positive for a long call or put |
| **Theta** | the daily cost of holding an option — the price paid for being long gamma | largest at the money and for short-dated options | usually negative for a long option |
| **Vega** | exposure to implied volatility; used to trade and hedge volatility | largest at the money and for long-dated options; near zero in the far wings | positive for a long call or put |
| **Rho** | exposure to interest rates | grows with maturity and with $K$; negligible for short-dated options | call: positive; put: negative |

Delta, Gamma and Theta are not independent: the Black–Scholes–Merton equation of §6.1.4,
$\Theta+\tfrac12\sigma^2S^2\Gamma+rS\Delta-rV=0$, ties them together (used in 6.2.3).

In [7]:
import math

def norm_pdf(x):
    """N'(x), the standard normal density."""
    return math.exp(-x * x / 2) / math.sqrt(2 * math.pi)

def d1_d2(S, K, r, sigma, tau, y=0.0):
    d1 = (math.log(S / K) + (r - y + sigma ** 2 / 2) * tau) / (sigma * math.sqrt(tau))
    return d1, d1 - sigma * math.sqrt(tau)

def greeks(S, K, r, sigma, tau, call=True, y=0.0):
    """Closed-form Greeks of a European option with continuous dividend yield y.
    theta = dV/dt = -dV/dtau (per year); vega is per 1.00 of volatility; rho is per 1.00 of rate.
    (N and bs are the normal cdf and Black-Scholes price defined in 6.1.1.)"""
    d1, d2 = d1_d2(S, K, r, sigma, tau, y)
    eq, er, sq = math.exp(-y * tau), math.exp(-r * tau), math.sqrt(tau)
    decay = -S * eq * norm_pdf(d1) * sigma / (2 * sq)                 # optionality decay (call and put)
    if call:
        delta = eq * N(d1)
        theta = decay + y * S * eq * N(d1) - r * K * er * N(d2)
        rho = K * tau * er * N(d2)
    else:
        delta = -eq * N(-d1)
        theta = decay - y * S * eq * N(-d1) + r * K * er * N(-d2)
        rho = -K * tau * er * N(-d2)
    gamma = eq * norm_pdf(d1) / (S * sigma * sq)                      # same for call and put
    vega = S * eq * norm_pdf(d1) * sq                                 # same for call and put
    return dict(delta=delta, gamma=gamma, theta=theta, vega=vega, rho=rho)

# check every closed form against a finite difference of the Black-Scholes price
S, K, r, sigma, tau, y, h = 100, 95, 0.05, 0.25, 0.5, 0.03, 1e-4
P = lambda call, **kw: bs(kw.get('S', S), K, kw.get('r', r), kw.get('sigma', sigma), kw.get('T', tau), call=call, q=y)
for call in (True, False):
    g = greeks(S, K, r, sigma, tau, call, y)
    fd = dict(delta=(P(call, S=S + h) - P(call, S=S - h)) / (2 * h),
              gamma=(P(call, S=S + h) - 2 * P(call) + P(call, S=S - h)) / h ** 2,
              theta=-(P(call, T=tau + h) - P(call, T=tau - h)) / (2 * h),
              vega=(P(call, sigma=sigma + h) - P(call, sigma=sigma - h)) / (2 * h),
              rho=(P(call, r=r + h) - P(call, r=r - h)) / (2 * h))
    print("CALL" if call else "PUT ", "  closed form  vs  finite difference")
    for k in ("delta", "gamma", "theta", "vega", "rho"):
        print(f"   {k:>5}: {g[k]:>10.5f}   {fd[k]:>10.5f}")

CALL   closed form  vs  finite difference
   delta:    0.65831      0.65831
   gamma:    0.02022      0.02022
   theta:   -7.13351     -7.13351
    vega:   25.27954     25.27954
     rho:   27.88562     27.88562
PUT    closed form  vs  finite difference
   delta:   -0.32680     -0.32680
   gamma:    0.02022      0.02022
   theta:   -5.45613     -5.45613
    vega:   25.27954     25.27954
     rho:  -18.44160    -18.44160


### 6.2.1 Delta

**General formula (continuous dividend yield $y$).** With
$d_1=\dfrac{\ln(S/K)+(r-y+\sigma^{2}/2)\tau}{\sigma\sqrt\tau}$ and $d_2=d_1-\sigma\sqrt\tau$,
$$\Delta_{\text{call}}=\frac{\partial c}{\partial S}=e^{-y\tau}N(d_1),\qquad
\Delta_{\text{put}}=\frac{\partial p}{\partial S}=e^{-y\tau}\big(N(d_1)-1\big)=-e^{-y\tau}N(-d_1).$$
A call's delta lies between $0$ and $e^{-y\tau}$, a put's between $-e^{-y\tau}$ and $0$, and
$\Delta_{\text{call}}-\Delta_{\text{put}}=e^{-y\tau}$ (put–call parity differentiated in $S$).

#### D1 — derive the delta of a European call (no dividends)

Start from $c=S\,N(d_1)-Ke^{-r\tau}N(d_2)$ with $d_1=\dfrac{\ln(S/K)+(r+\sigma^2/2)\tau}{\sigma\sqrt\tau}$,
$d_2=d_1-\sigma\sqrt\tau$, and $N'(x)=\dfrac{1}{\sqrt{2\pi}}e^{-x^{2}/2}$.

**Step 1 — product rule.** $S$ appears both as the explicit factor in front of $N(d_1)$ and inside $d_1,d_2$:
$$\frac{\partial c}{\partial S}=N(d_1)+S\,\frac{\partial N(d_1)}{\partial S}-Ke^{-r\tau}\,\frac{\partial N(d_2)}{\partial S}.$$

**Step 2 — differentiate $d_1$ and $d_2$ with respect to $S$.** In $d_1$ only the term $\ln S$ depends on $S$, so
$$\frac{\partial d_1}{\partial S}=\frac{1}{S\,\sigma\sqrt\tau},\qquad
\frac{\partial d_2}{\partial S}=\frac{\partial d_1}{\partial S}=\frac{1}{S\,\sigma\sqrt\tau}\quad(\text{since }d_2=d_1-\sigma\sqrt\tau\text{ differs by a constant}).$$

**Step 3 — chain rule for $N(d_1)$ and $N(d_2)$.** Because $\frac{d}{dx}N(x)=N'(x)$,
$$\frac{\partial N(d_1)}{\partial S}=N'(d_1)\,\frac{1}{S\sigma\sqrt\tau},\qquad
\frac{\partial N(d_2)}{\partial S}=N'(d_2)\,\frac{1}{S\sigma\sqrt\tau}.$$
Substituting both into Step 1,
$$\frac{\partial c}{\partial S}=N(d_1)+\frac{S\,N'(d_1)-Ke^{-r\tau}N'(d_2)}{S\,\sigma\sqrt\tau}.$$

**Step 4 — the key identity $S\,N'(d_1)=Ke^{-r\tau}N'(d_2)$.** Since $d_2=d_1-\sigma\sqrt\tau$,
$$N'(d_2)=\frac{1}{\sqrt{2\pi}}e^{-(d_1-\sigma\sqrt\tau)^{2}/2}=N'(d_1)\,e^{\,d_1\sigma\sqrt\tau-\sigma^{2}\tau/2}.$$
Now $d_1\sigma\sqrt\tau=\ln(S/K)+(r+\sigma^{2}/2)\tau$, so the exponent equals $\ln(S/K)+r\tau$ and
$$N'(d_2)=N'(d_1)\,\frac{S}{K}\,e^{r\tau}\quad\Longleftrightarrow\quad Ke^{-r\tau}N'(d_2)=S\,N'(d_1).$$

**Step 5 — the extra terms cancel.** The numerator in Step 3 is exactly $S\,N'(d_1)-Ke^{-r\tau}N'(d_2)=0$, so
$$\boxed{\Delta_{\text{call}}=N(d_1)}.$$
For the put, parity $p=c-S+Ke^{-r\tau}$ gives $\Delta_{\text{put}}=N(d_1)-1$. (Reading the result: $N(d_1)$ is
the number of shares that replicates the call; its sibling $N(d_2)$ is the risk-neutral probability that the
call finishes in the money.)

In [8]:
# Step 4 identity and the result Delta = N(d1), checked against a finite difference of the price
S, K, r, sigma, tau = 100, 95, 0.05, 0.25, 0.5
d1, d2 = d1_d2(S, K, r, sigma, tau)
print(f"identity:  S N'(d1) = {S * norm_pdf(d1):.8f}    K e^(-r tau) N'(d2) = {K * math.exp(-r * tau) * norm_pdf(d2):.8f}")
h = 1e-4
fd_call = (bs(S + h, K, r, sigma, tau) - bs(S - h, K, r, sigma, tau)) / (2 * h)
fd_put = (bs(S + h, K, r, sigma, tau, call=False) - bs(S - h, K, r, sigma, tau, call=False)) / (2 * h)
print(f"call delta:  N(d1)   = {N(d1):.6f}    finite difference = {fd_call:.6f}")
print(f"put delta:   N(d1)-1 = {N(d1) - 1:.6f}   finite difference = {fd_put:.6f}")
print(f"N(d2) = {N(d2):.4f} is the risk-neutral probability the call finishes in the money")

identity:  S N'(d1) = 34.84982401    K e^(-r tau) N'(d2) = 34.84982401
call delta:  N(d1)   = 0.698457    finite difference = 0.698457
put delta:   N(d1)-1 = -0.301543   finite difference = -0.301543
N(d2) = 0.6343 is the risk-neutral probability the call finishes in the money


#### D2 — delta of an at-the-money call, and what happens near maturity

At the money, $S=K$, so $\ln(S/K)=0$ and
$$d_1=\frac{(r+\sigma^{2}/2)\,\tau}{\sigma\sqrt\tau}=\Big(\frac{r}{\sigma}+\frac{\sigma}{2}\Big)\sqrt\tau ,$$
a small number for a short- or medium-dated option. Linearise $N$ around $0$ using $N(0)=\tfrac12$ and
$N'(0)=\tfrac{1}{\sqrt{2\pi}}\approx0.4$:
$$\Delta_{\text{ATM call}}=N(d_1)\;\approx\;\tfrac12+0.4\,d_1=\tfrac12+0.4\Big(\frac r\sigma+\frac\sigma2\Big)\sqrt\tau\;\approx\;\boxed{0.5}.$$
It sits just above $0.5$: the positive drift $r+\sigma^2/2$ makes finishing in the money slightly more likely
than not. (For $K=100$, $r=5\%$, $\sigma=25\%$ and $\tau=3$ months the formula gives $0.565$.)

**As maturity approaches.** For exactly $S=K$ the estimate shows $d_1\to0$ as $\tau\to0$, so the ATM delta tends
to $0.5$. But the **entire curve sharpens**. For $S\neq K$ we have $d_1\approx\dfrac{\ln(S/K)}{\sigma\sqrt\tau}\to\pm\infty$, so
$$\Delta_{\text{call}}\to\begin{cases}1,&S>K\ \text{(certain to finish in the money)},\\[2pt]0,&S<K\ \text{(certain to finish out of the money)},\end{cases}$$
a **step function** at the strike. Near expiry a tiny move across $K$ swings delta from almost $0$ to almost
$1$, so the delta of a near-the-money option becomes extremely unstable (the subject of Gamma, 6.2.2).

<p align="center">
<svg viewBox="0 0 520 330" width="520" xmlns="http://www.w3.org/2000/svg" style="max-width:100%;height:auto" font-family="sans-serif">
<line x1="66" y1="278.0" x2="504" y2="278.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="282.0" font-size="11" fill="currentColor" text-anchor="end">0.0</text>
<line x1="66" y1="225.2" x2="504" y2="225.2" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="229.2" font-size="11" fill="currentColor" text-anchor="end">0.2</text>
<line x1="66" y1="172.4" x2="504" y2="172.4" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="176.4" font-size="11" fill="currentColor" text-anchor="end">0.4</text>
<line x1="66" y1="119.6" x2="504" y2="119.6" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="123.6" font-size="11" fill="currentColor" text-anchor="end">0.6</text>
<line x1="66" y1="66.8" x2="504" y2="66.8" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="70.8" font-size="11" fill="currentColor" text-anchor="end">0.8</text>
<line x1="66" y1="14.0" x2="504" y2="14.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="18.0" font-size="11" fill="currentColor" text-anchor="end">1.0</text>
<line x1="66.0" y1="14" x2="66.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="66.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">60</text>
<line x1="175.5" y1="14" x2="175.5" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="175.5" y="294" font-size="11" fill="currentColor" text-anchor="middle">80</text>
<line x1="285.0" y1="14" x2="285.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="285.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">100</text>
<line x1="394.5" y1="14" x2="394.5" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="394.5" y="294" font-size="11" fill="currentColor" text-anchor="middle">120</text>
<line x1="504.0" y1="14" x2="504.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="504.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">140</text>
<rect x="66" y="14" width="438" height="264" fill="none" stroke="currentColor" stroke-opacity="0.45"/>
<line x1="285.0" y1="14" x2="285.0" y2="278" stroke="currentColor" stroke-opacity="0.5" stroke-dasharray="4 3"/>
<text x="289.0" y="26" font-size="11" fill="currentColor" fill-opacity="0.8">K = 100</text>
<polyline points="66.0,278.0 68.7,278.0 71.5,278.0 74.2,278.0 77.0,278.0 79.7,278.0 82.4,278.0 85.2,278.0 87.9,278.0 90.6,278.0 93.4,278.0 96.1,278.0 98.8,278.0 101.6,278.0 104.3,278.0 107.1,278.0 109.8,278.0 112.5,278.0 115.3,278.0 118.0,278.0 120.8,278.0 123.5,278.0 126.2,278.0 129.0,278.0 131.7,278.0 134.4,278.0 137.2,278.0 139.9,278.0 142.6,278.0 145.4,278.0 148.1,278.0 150.9,278.0 153.6,278.0 156.3,278.0 159.1,278.0 161.8,278.0 164.6,278.0 167.3,278.0 170.0,278.0 172.8,278.0 175.5,278.0 178.2,278.0 181.0,278.0 183.7,278.0 186.4,278.0 189.2,278.0 191.9,278.0 194.7,278.0 197.4,278.0 200.1,278.0 202.9,278.0 205.6,278.0 208.3,278.0 211.1,277.9 213.8,277.9 216.6,277.8 219.3,277.7 222.0,277.5 224.8,277.2 227.5,276.9 230.2,276.3 233.0,275.6 235.7,274.6 238.5,273.2 241.2,271.4 243.9,269.1 246.7,266.2 249.4,262.6 252.2,258.3 254.9,253.0 257.6,246.9 260.4,239.8 263.1,231.7 265.8,222.6 268.6,212.7 271.3,201.9 274.0,190.4 276.8,178.3 279.5,165.8 282.3,153.1 285.0,140.3 287.7,127.7 290.5,115.5 293.2,103.7 296.0,92.5 298.7,82.0 301.4,72.4 304.2,63.6 306.9,55.8 309.6,48.8 312.4,42.7 315.1,37.5 317.9,33.0 320.6,29.2 323.3,26.0 326.1,23.5 328.8,21.3 331.5,19.7 334.3,18.3 337.0,17.3 339.8,16.4 342.5,15.8 345.2,15.3 348.0,15.0 350.7,14.7 353.4,14.5 356.2,14.3 358.9,14.2 361.7,14.2 364.4,14.1 367.1,14.1 369.9,14.1 372.6,14.0 375.3,14.0 378.1,14.0 380.8,14.0 383.6,14.0 386.3,14.0 389.0,14.0 391.8,14.0 394.5,14.0 397.2,14.0 400.0,14.0 402.7,14.0 405.4,14.0 408.2,14.0 410.9,14.0 413.7,14.0 416.4,14.0 419.1,14.0 421.9,14.0 424.6,14.0 427.3,14.0 430.1,14.0 432.8,14.0 435.6,14.0 438.3,14.0 441.0,14.0 443.8,14.0 446.5,14.0 449.2,14.0 452.0,14.0 454.7,14.0 457.5,14.0 460.2,14.0 462.9,14.0 465.7,14.0 468.4,14.0 471.2,14.0 473.9,14.0 476.6,14.0 479.4,14.0 482.1,14.0 484.8,14.0 487.6,14.0 490.3,14.0 493.1,14.0 495.8,14.0 498.5,14.0 501.3,14.0 504.0,14.0" fill="none" stroke="#3d6ea5" stroke-width="2.2" stroke-linejoin="round"/>
<polyline points="66.0,278.0 68.7,278.0 71.5,278.0 74.2,278.0 77.0,278.0 79.7,278.0 82.4,278.0 85.2,278.0 87.9,278.0 90.6,278.0 93.4,278.0 96.1,278.0 98.8,278.0 101.6,278.0 104.3,278.0 107.1,278.0 109.8,278.0 112.5,278.0 115.3,278.0 118.0,278.0 120.8,278.0 123.5,278.0 126.2,278.0 129.0,278.0 131.7,278.0 134.4,278.0 137.2,278.0 139.9,278.0 142.6,278.0 145.4,278.0 148.1,278.0 150.9,278.0 153.6,278.0 156.3,278.0 159.1,277.9 161.8,277.9 164.6,277.9 167.3,277.9 170.0,277.8 172.8,277.7 175.5,277.6 178.2,277.5 181.0,277.4 183.7,277.2 186.4,277.0 189.2,276.7 191.9,276.3 194.7,275.9 197.4,275.3 200.1,274.7 202.9,273.9 205.6,273.0 208.3,271.9 211.1,270.7 213.8,269.2 216.6,267.6 219.3,265.7 222.0,263.5 224.8,261.1 227.5,258.3 230.2,255.3 233.0,252.0 235.7,248.3 238.5,244.3 241.2,239.9 243.9,235.2 246.7,230.2 249.4,224.9 252.2,219.2 254.9,213.3 257.6,207.1 260.4,200.6 263.1,193.9 265.8,187.0 268.6,180.0 271.3,172.8 274.0,165.5 276.8,158.1 279.5,150.8 282.3,143.4 285.0,136.1 287.7,128.9 290.5,121.8 293.2,114.9 296.0,108.1 298.7,101.5 301.4,95.1 304.2,89.0 306.9,83.2 309.6,77.6 312.4,72.3 315.1,67.2 317.9,62.5 320.6,58.1 323.3,53.9 326.1,50.1 328.8,46.5 331.5,43.2 334.3,40.1 337.0,37.3 339.8,34.8 342.5,32.4 345.2,30.3 348.0,28.4 350.7,26.7 353.4,25.1 356.2,23.8 358.9,22.5 361.7,21.4 364.4,20.4 367.1,19.6 369.9,18.8 372.6,18.2 375.3,17.6 378.1,17.1 380.8,16.6 383.6,16.2 386.3,15.9 389.0,15.6 391.8,15.4 394.5,15.2 397.2,15.0 400.0,14.8 402.7,14.7 405.4,14.6 408.2,14.5 410.9,14.4 413.7,14.3 416.4,14.3 419.1,14.2 421.9,14.2 424.6,14.2 427.3,14.1 430.1,14.1 432.8,14.1 435.6,14.1 438.3,14.1 441.0,14.0 443.8,14.0 446.5,14.0 449.2,14.0 452.0,14.0 454.7,14.0 457.5,14.0 460.2,14.0 462.9,14.0 465.7,14.0 468.4,14.0 471.2,14.0 473.9,14.0 476.6,14.0 479.4,14.0 482.1,14.0 484.8,14.0 487.6,14.0 490.3,14.0 493.1,14.0 495.8,14.0 498.5,14.0 501.3,14.0 504.0,14.0" fill="none" stroke="#e69138" stroke-width="2.2" stroke-linejoin="round"/>
<polyline points="66.0,278.0 68.7,278.0 71.5,278.0 74.2,278.0 77.0,278.0 79.7,278.0 82.4,277.9 85.2,277.9 87.9,277.9 90.6,277.9 93.4,277.9 96.1,277.8 98.8,277.8 101.6,277.7 104.3,277.7 107.1,277.6 109.8,277.5 112.5,277.4 115.3,277.3 118.0,277.2 120.8,277.1 123.5,276.9 126.2,276.7 129.0,276.5 131.7,276.2 134.4,275.9 137.2,275.6 139.9,275.2 142.6,274.7 145.4,274.3 148.1,273.7 150.9,273.1 153.6,272.4 156.3,271.7 159.1,270.9 161.8,270.0 164.6,269.0 167.3,268.0 170.0,266.8 172.8,265.5 175.5,264.2 178.2,262.7 181.0,261.1 183.7,259.5 186.4,257.7 189.2,255.7 191.9,253.7 194.7,251.5 197.4,249.2 200.1,246.8 202.9,244.3 205.6,241.6 208.3,238.9 211.1,236.0 213.8,232.9 216.6,229.8 219.3,226.6 222.0,223.2 224.8,219.7 227.5,216.2 230.2,212.5 233.0,208.7 235.7,204.9 238.5,201.0 241.2,197.0 243.9,192.9 246.7,188.8 249.4,184.6 252.2,180.4 254.9,176.1 257.6,171.8 260.4,167.5 263.1,163.2 265.8,158.9 268.6,154.5 271.3,150.2 274.0,145.9 276.8,141.6 279.5,137.4 282.3,133.1 285.0,129.0 287.7,124.8 290.5,120.7 293.2,116.7 296.0,112.8 298.7,108.9 301.4,105.1 304.2,101.3 306.9,97.7 309.6,94.1 312.4,90.6 315.1,87.2 317.9,83.9 320.6,80.7 323.3,77.6 326.1,74.5 328.8,71.6 331.5,68.8 334.3,66.0 337.0,63.4 339.8,60.9 342.5,58.4 345.2,56.1 348.0,53.8 350.7,51.6 353.4,49.5 356.2,47.6 358.9,45.6 361.7,43.8 364.4,42.1 367.1,40.4 369.9,38.9 372.6,37.4 375.3,35.9 378.1,34.6 380.8,33.3 383.6,32.1 386.3,30.9 389.0,29.9 391.8,28.8 394.5,27.9 397.2,26.9 400.0,26.1 402.7,25.3 405.4,24.5 408.2,23.8 410.9,23.1 413.7,22.5 416.4,21.9 419.1,21.3 421.9,20.8 424.6,20.3 427.3,19.8 430.1,19.4 432.8,19.0 435.6,18.6 438.3,18.3 441.0,18.0 443.8,17.7 446.5,17.4 449.2,17.1 452.0,16.9 454.7,16.7 457.5,16.5 460.2,16.3 462.9,16.1 465.7,15.9 468.4,15.8 471.2,15.6 473.9,15.5 476.6,15.4 479.4,15.3 482.1,15.2 484.8,15.1 487.6,15.0 490.3,14.9 493.1,14.8 495.8,14.7 498.5,14.7 501.3,14.6 504.0,14.6" fill="none" stroke="#6aa84f" stroke-width="2.2" stroke-linejoin="round"/>
<line x1="76" y1="30" x2="98" y2="30" stroke="#3d6ea5" stroke-width="2.6"/>
<text x="104" y="34" font-size="11.5" fill="currentColor">10 days</text>
<line x1="76" y1="48" x2="98" y2="48" stroke="#e69138" stroke-width="2.6"/>
<text x="104" y="52" font-size="11.5" fill="currentColor">1 month</text>
<line x1="76" y1="66" x2="98" y2="66" stroke="#6aa84f" stroke-width="2.6"/>
<text x="104" y="70" font-size="11.5" fill="currentColor">3 months</text>
<text x="285" y="320" font-size="12.5" fill="currentColor" text-anchor="middle">spot price S</text>
<text x="16" y="146" font-size="12.5" fill="currentColor" text-anchor="middle" transform="rotate(-90 16 146)">call delta</text>
</svg>
</p>

*Call delta against spot for 10 days, 1 month and 3 months to maturity ($K=100$, $r=5\%$, $\sigma=25\%$):
the shorter the maturity, the closer the curve is to a step at $K=100$, while all three cross near $0.5$ at the
strike.*

In [9]:
# ATM delta estimate 0.5 + 0.4 (r/sigma + sigma/2) sqrt(tau)  vs exact, and the sharpening of the curve near expiry
K, r, sigma = 100, 0.05, 0.25
mats = [("10 days", 10 / 365), ("1 month", 1 / 12), ("3 months", 0.25)]
print("ATM delta:  estimate   exact")
for lab, T in mats:
    est = 0.5 + 0.4 * (r / sigma + sigma / 2) * math.sqrt(T)
    print(f"  {lab:>8}:   {est:.4f}   {greeks(K, K, r, sigma, T)['delta']:.4f}")
print("\ncall delta at spot 90 / 95 / 100 / 105 / 110  (curve steepens as maturity shrinks):")
for lab, T in mats:
    print(f"  {lab:>8}: " + "  ".join(f"{greeks(s, K, r, sigma, T)['delta']:.3f}" for s in (90, 95, 100, 105, 110)))
print("\nextreme case, 1 hour before expiry (spot 99.5 / 100 / 100.5):  ",
      "  ".join(f"{greeks(s, K, r, sigma, 1 / 365 / 24)['delta']:.3f}" for s in (99.5, 100, 100.5)))

ATM delta:  estimate   exact
   10 days:   0.5215   0.5215
   1 month:   0.5375   0.5374
  3 months:   0.5650   0.5645

call delta at spot 90 / 95 / 100 / 105 / 110  (curve steepens as maturity shrinks):
   10 days: 0.006  0.118  0.521  0.891  0.991
   1 month: 0.086  0.269  0.537  0.779  0.921
  3 months: 0.248  0.402  0.565  0.710  0.823

extreme case, 1 hour before expiry (spot 99.5 / 100 / 100.5):   0.031  0.501  0.969


#### D3 — dynamically delta-hedging a long call on GM

**The hedge.** The long call gains $\Delta$ dollars for each 1-dollar rise in GM, so to cancel that exposure
**short $\Delta=N(d_1)$ shares of GM per call** (for a standard 100-share contract, short $100\,\Delta$ shares),
funded with cash. The hedged portfolio $\Pi=c-\Delta S$ has no first-order sensitivity to $S$:
$$d\Pi\approx\Delta\,dS-\Delta\,dS+\tfrac12\Gamma\,(dS)^{2}+\Theta\,dt=\tfrac12\Gamma\,(dS)^{2}+\Theta\,dt .$$

*Example.* GM at $100$, strike $100$, $r=5\%$, $\sigma=25\%$, three months to expiry: the call is worth
$c=5.60$ with $\Delta=0.5645$, so short $0.5645$ shares per call ($56.45$ shares per 100-share contract).

**Rebalancing after a sudden rise.** The hedge is exact only for infinitesimal moves. When GM jumps up the call's
delta *rises* (the call has positive gamma), so the existing short is now too small — the portfolio has become net
**long**. Restore delta-neutrality by **shorting additional shares**:
$$\text{extra shares to short}\;\approx\;\Gamma\,\Delta S .$$
Here a jump from $100$ to $103$ lifts delta from $0.5645$ to $0.6550$, so short about $0.09$ more shares (the
first-order estimate $\Gamma\cdot\Delta S=0.0945$). After a *fall* do the reverse: buy shares back. The rule is
therefore **sell into rallies, buy into dips**, which is how a long-gamma hedger earns the profit that pays for
the option's time decay.

**Why dynamic?** Delta changes with $S$ and with time, so the hedge must be adjusted repeatedly. The simulation
below hedges a call on a stock with real-world drift $12\%$ and shows the hedge error shrinking roughly like
$1/\sqrt{n}$ as the number of rebalances $n$ grows (costs of trading are what limit how large $n$ should be in
practice).

In [10]:
import math, random

S0, K, r, sigma, T = 100, 100, 0.05, 0.25, 0.25
g0 = greeks(S0, K, r, sigma, T)
print(f"call price {bs(S0, K, r, sigma, T):.4f},  delta {g0['delta']:.4f}  ->  short {g0['delta']:.4f} shares per call "
      f"({100 * g0['delta']:.2f} per 100-share contract)")
g1 = greeks(103, K, r, sigma, T)
print(f"GM jumps 100 -> 103: delta {g0['delta']:.4f} -> {g1['delta']:.4f};  short {g1['delta'] - g0['delta']:.4f} more shares "
      f"(first-order estimate Gamma*dS = {g0['gamma'] * 3:.4f})")

def hedge_pnl(n, paths=700, mu=0.12, seed=1):
    """Final P&L of: long 1 call, short delta shares, rebalanced n times; cash earns r.  Ideal hedge -> 0."""
    rng = random.Random(seed); dt = T / n; out = []
    for _ in range(paths):
        S = S0; dl = greeks(S0, K, r, sigma, T)['delta']
        cash = dl * S0 - bs(S0, K, r, sigma, T)              # short shares bring cash in; the call premium is paid
        for i in range(1, n + 1):
            S *= math.exp((mu - sigma ** 2 / 2) * dt + sigma * math.sqrt(dt) * rng.gauss(0, 1))
            cash *= math.exp(r * dt)
            if i < n:
                nd = greeks(S, K, r, sigma, T - i * dt)['delta']
                cash += (nd - dl) * S; dl = nd                # sell (nd - dl) more shares
        out.append(max(S - K, 0) - dl * S + cash)             # call payoff, buy back shares, plus cash
    m = sum(out) / paths
    return m, (sum((x - m) ** 2 for x in out) / paths) ** 0.5

print("\nhedge error vs number of rebalances (call on a stock drifting at 12%; ideal P&L = 0):")
for n in (1, 4, 16, 64, 252):
    m, sd = hedge_pnl(n)
    print(f"  n = {n:>3}:  mean P&L {m:+.3f},  std of P&L {sd:.3f}")

call price 5.5984,  delta 0.5645  ->  short 0.5645 shares per call (56.45 per 100-share contract)
GM jumps 100 -> 103: delta 0.5645 -> 0.6550;  short 0.0905 more shares (first-order estimate Gamma*dS = 0.0945)

hedge error vs number of rebalances (call on a stock drifting at 12%; ideal P&L = 0):
  n =   1:  mean P&L -0.043,  std of P&L 3.674
  n =   4:  mean P&L +0.016,  std of P&L 1.930
  n =  16:  mean P&L -0.049,  std of P&L 1.035
  n =  64:  mean P&L +0.024,  std of P&L 0.519
  n = 252:  mean P&L +0.009,  std of P&L 0.268


#### D4 — estimating the value of an at-the-money call

Take a non-dividend stock at the money ($S=K$) with a low interest rate and a short maturity. Then
$d_1=\tfrac12\sigma\sqrt\tau$, $d_2=-\tfrac12\sigma\sqrt\tau$ and $Ke^{-r\tau}\approx S$, so
$$c=S\,N(d_1)-Ke^{-r\tau}N(d_2)\;\approx\;S\Big[N\big(\tfrac{\sigma\sqrt\tau}{2}\big)-N\big(-\tfrac{\sigma\sqrt\tau}{2}\big)\Big].$$
For $x$ small, $N(x)-N(-x)=\int_{-x}^{x}N'(u)\,du\approx2x\,N'(0)$, because the density is nearly flat at
$N'(0)=\dfrac{1}{\sqrt{2\pi}}$. With $x=\tfrac12\sigma\sqrt\tau$,
$$\boxed{\,c\;\approx\;\frac{1}{\sqrt{2\pi}}\,S\,\sigma\sqrt\tau\;\approx\;0.4\,S\,\sigma\sqrt\tau\,}.$$
A handy rule of thumb: **0.4 $\times$ stock price $\times$ volatility $\times\sqrt{\text{time}}$**. For example
$S=100$, $\sigma=25\%$, one month ($\tau=\tfrac1{12}$): $0.4\times100\times0.25\times0.289\approx2.9$. With a
modest positive rate, add roughly $\tfrac12\,r\,S\,\tau$ (the interest saved on deferring the strike): the
second term here is $0.21$, taking the estimate to about $3.1$.

In [11]:
# ATM call: exact vs 0.4 S sigma sqrt(tau)  (+ 0.5 r S tau correction when r > 0)
S = K = 100
sigma = 0.25
print("maturity   exact(r=0)  0.4*S*sigma*sqrt(tau)   exact(r=5%)  estimate + 0.5*r*S*tau")
for lab, T in [("10 days", 10 / 365), ("1 month", 1 / 12), ("3 months", 0.25)]:
    c0 = bs(S, K, 0.0, sigma, T)
    est = 0.4 * S * sigma * math.sqrt(T)
    c5 = bs(S, K, 0.05, sigma, T)
    print(f"{lab:>9}   {c0:8.4f}    {est:8.4f}                {c5:8.4f}    {est + 0.5 * 0.05 * S * T:8.4f}")

maturity   exact(r=0)  0.4*S*sigma*sqrt(tau)   exact(r=5%)  estimate + 0.5*r*S*tau
  10 days     1.6507      1.6552                  1.7189      1.7237
  1 month     2.8785      2.8868                  3.0852      3.0951
 3 months     4.9835      5.0000                  5.5984      5.6250


### 6.2.2 Gamma

**General formula (continuous dividend yield $y$).** Gamma is the second derivative of the price in $S$ — the rate
of change of delta:
$$\Gamma=\frac{\partial^{2}V}{\partial S^{2}}=\frac{\partial\Delta}{\partial S}=\frac{e^{-y\tau}\,N'(d_1)}{S\,\sigma\sqrt\tau}.$$
It follows by differentiating $\Delta_{\text{call}}=e^{-y\tau}N(d_1)$ once more using
$\partial d_1/\partial S=1/(S\sigma\sqrt\tau)$. The **put has the same gamma as the call** with the same strike and
maturity, because $\Delta_{\text{call}}-\Delta_{\text{put}}=e^{-y\tau}$ does not depend on $S$.

#### G1 — the gamma of an at-the-money option as maturity approaches

At the money $d_1=\big(\tfrac r\sigma+\tfrac\sigma2\big)\sqrt\tau\approx0$, so $N'(d_1)\approx N'(0)\approx0.4$ and
$$\Gamma_{\text{ATM}}\;\approx\;\frac{0.4\,e^{-y\tau}}{S\,\sigma\sqrt\tau}\;\xrightarrow[\tau\to0]{}\;\infty,$$
growing like $1/\sqrt\tau$: **an at-the-money option's gamma blows up at expiry.** This is the step function of
D2 seen from the second-derivative side — delta must jump from $0$ to $1$ across the strike, and the slope of a
near-vertical jump is enormous. Away from the strike the opposite happens: $d_1\to\pm\infty$, $N'(d_1)\to0$
exponentially and gamma collapses to $0$. So as $\tau$ shrinks the gamma curve becomes a **taller, narrower spike**
(its width is of order $S\sigma\sqrt\tau$) centred at the strike.

Practical consequence ("gamma risk" or "pin risk"): a near-expiry ATM option needs constant rehedging, and a
delta hedge can be badly wrong after a small move — which is why long-gamma positions are valuable but expensive
(they pay it back as time decay, 6.2.3).

<p align="center">
<svg viewBox="0 0 520 330" width="520" xmlns="http://www.w3.org/2000/svg" style="max-width:100%;height:auto" font-family="sans-serif">
<line x1="66" y1="278.0" x2="504" y2="278.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="282.0" font-size="11" fill="currentColor" text-anchor="end">0.00</text>
<line x1="66" y1="225.2" x2="504" y2="225.2" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="229.2" font-size="11" fill="currentColor" text-anchor="end">0.02</text>
<line x1="66" y1="172.4" x2="504" y2="172.4" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="176.4" font-size="11" fill="currentColor" text-anchor="end">0.04</text>
<line x1="66" y1="119.6" x2="504" y2="119.6" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="123.6" font-size="11" fill="currentColor" text-anchor="end">0.06</text>
<line x1="66" y1="66.8" x2="504" y2="66.8" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="70.8" font-size="11" fill="currentColor" text-anchor="end">0.08</text>
<line x1="66" y1="14.0" x2="504" y2="14.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="18.0" font-size="11" fill="currentColor" text-anchor="end">0.10</text>
<line x1="66.0" y1="14" x2="66.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="66.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">60</text>
<line x1="175.5" y1="14" x2="175.5" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="175.5" y="294" font-size="11" fill="currentColor" text-anchor="middle">80</text>
<line x1="285.0" y1="14" x2="285.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="285.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">100</text>
<line x1="394.5" y1="14" x2="394.5" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="394.5" y="294" font-size="11" fill="currentColor" text-anchor="middle">120</text>
<line x1="504.0" y1="14" x2="504.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="504.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">140</text>
<rect x="66" y="14" width="438" height="264" fill="none" stroke="currentColor" stroke-opacity="0.45"/>
<line x1="285.0" y1="14" x2="285.0" y2="278" stroke="currentColor" stroke-opacity="0.5" stroke-dasharray="4 3"/>
<text x="289.0" y="26" font-size="11" fill="currentColor" fill-opacity="0.8">K = 100</text>
<polyline points="66.0,278.0 68.7,278.0 71.5,278.0 74.2,278.0 77.0,278.0 79.7,278.0 82.4,278.0 85.2,278.0 87.9,278.0 90.6,278.0 93.4,278.0 96.1,278.0 98.8,278.0 101.6,278.0 104.3,278.0 107.1,278.0 109.8,278.0 112.5,278.0 115.3,278.0 118.0,278.0 120.8,278.0 123.5,278.0 126.2,278.0 129.0,278.0 131.7,278.0 134.4,278.0 137.2,278.0 139.9,278.0 142.6,278.0 145.4,278.0 148.1,278.0 150.9,278.0 153.6,278.0 156.3,278.0 159.1,278.0 161.8,278.0 164.6,278.0 167.3,278.0 170.0,278.0 172.8,278.0 175.5,278.0 178.2,278.0 181.0,278.0 183.7,278.0 186.4,278.0 189.2,278.0 191.9,278.0 194.7,278.0 197.4,277.9 200.1,277.9 202.9,277.8 205.6,277.7 208.3,277.5 211.1,277.2 213.8,276.8 216.6,276.1 219.3,275.1 222.0,273.7 224.8,271.7 227.5,269.0 230.2,265.3 233.0,260.6 235.7,254.5 238.5,246.9 241.2,237.6 243.9,226.5 246.7,213.5 249.4,198.7 252.2,182.2 254.9,164.3 257.6,145.4 260.4,125.9 263.1,106.4 265.8,87.6 268.6,70.1 271.3,54.6 274.0,41.7 276.8,31.8 279.5,25.5 282.3,22.8 285.0,23.8 287.7,28.6 290.5,36.7 293.2,47.8 296.0,61.4 298.7,77.0 301.4,94.0 304.2,111.8 306.9,129.8 309.6,147.6 312.4,164.6 315.1,180.7 317.9,195.5 320.6,208.9 323.3,220.9 326.1,231.3 328.8,240.2 331.5,247.8 334.3,254.2 337.0,259.4 339.8,263.6 342.5,267.0 345.2,269.7 348.0,271.8 350.7,273.4 353.4,274.6 356.2,275.6 358.9,276.2 361.7,276.7 364.4,277.1 367.1,277.4 369.9,277.6 372.6,277.7 375.3,277.8 378.1,277.9 380.8,277.9 383.6,277.9 386.3,278.0 389.0,278.0 391.8,278.0 394.5,278.0 397.2,278.0 400.0,278.0 402.7,278.0 405.4,278.0 408.2,278.0 410.9,278.0 413.7,278.0 416.4,278.0 419.1,278.0 421.9,278.0 424.6,278.0 427.3,278.0 430.1,278.0 432.8,278.0 435.6,278.0 438.3,278.0 441.0,278.0 443.8,278.0 446.5,278.0 449.2,278.0 452.0,278.0 454.7,278.0 457.5,278.0 460.2,278.0 462.9,278.0 465.7,278.0 468.4,278.0 471.2,278.0 473.9,278.0 476.6,278.0 479.4,278.0 482.1,278.0 484.8,278.0 487.6,278.0 490.3,278.0 493.1,278.0 495.8,278.0 498.5,278.0 501.3,278.0 504.0,278.0" fill="none" stroke="#3d6ea5" stroke-width="2.2" stroke-linejoin="round"/>
<polyline points="66.0,278.0 68.7,278.0 71.5,278.0 74.2,278.0 77.0,278.0 79.7,278.0 82.4,278.0 85.2,278.0 87.9,278.0 90.6,278.0 93.4,278.0 96.1,278.0 98.8,278.0 101.6,278.0 104.3,278.0 107.1,278.0 109.8,278.0 112.5,278.0 115.3,278.0 118.0,278.0 120.8,278.0 123.5,278.0 126.2,278.0 129.0,278.0 131.7,278.0 134.4,278.0 137.2,278.0 139.9,278.0 142.6,278.0 145.4,277.9 148.1,277.9 150.9,277.9 153.6,277.8 156.3,277.7 159.1,277.6 161.8,277.5 164.6,277.3 167.3,277.1 170.0,276.8 172.8,276.4 175.5,276.0 178.2,275.4 181.0,274.7 183.7,273.8 186.4,272.8 189.2,271.5 191.9,270.0 194.7,268.3 197.4,266.3 200.1,263.9 202.9,261.3 205.6,258.2 208.3,254.9 211.1,251.1 213.8,246.9 216.6,242.3 219.3,237.4 222.0,232.1 224.8,226.4 227.5,220.5 230.2,214.2 233.0,207.8 235.7,201.2 238.5,194.4 241.2,187.7 243.9,181.0 246.7,174.4 249.4,168.1 252.2,162.0 254.9,156.3 257.6,151.0 260.4,146.2 263.1,142.0 265.8,138.4 268.6,135.4 271.3,133.2 274.0,131.6 276.8,130.8 279.5,130.7 282.3,131.4 285.0,132.7 287.7,134.7 290.5,137.3 293.2,140.6 296.0,144.3 298.7,148.5 301.4,153.2 304.2,158.2 306.9,163.5 309.6,169.0 312.4,174.7 315.1,180.4 317.9,186.3 320.6,192.1 323.3,197.9 326.1,203.5 328.8,209.1 331.5,214.4 334.3,219.6 337.0,224.5 339.8,229.2 342.5,233.7 345.2,237.8 348.0,241.7 350.7,245.4 353.4,248.7 356.2,251.9 358.9,254.7 361.7,257.3 364.4,259.7 367.1,261.8 369.9,263.8 372.6,265.5 375.3,267.1 378.1,268.5 380.8,269.7 383.6,270.8 386.3,271.8 389.0,272.7 391.8,273.4 394.5,274.1 397.2,274.6 400.0,275.1 402.7,275.6 405.4,275.9 408.2,276.3 410.9,276.5 413.7,276.8 416.4,277.0 419.1,277.1 421.9,277.3 424.6,277.4 427.3,277.5 430.1,277.6 432.8,277.7 435.6,277.7 438.3,277.8 441.0,277.8 443.8,277.8 446.5,277.9 449.2,277.9 452.0,277.9 454.7,277.9 457.5,277.9 460.2,278.0 462.9,278.0 465.7,278.0 468.4,278.0 471.2,278.0 473.9,278.0 476.6,278.0 479.4,278.0 482.1,278.0 484.8,278.0 487.6,278.0 490.3,278.0 493.1,278.0 495.8,278.0 498.5,278.0 501.3,278.0 504.0,278.0" fill="none" stroke="#e69138" stroke-width="2.2" stroke-linejoin="round"/>
<polyline points="66.0,277.9 68.7,277.9 71.5,277.9 74.2,277.9 77.0,277.8 79.7,277.8 82.4,277.7 85.2,277.7 87.9,277.6 90.6,277.5 93.4,277.4 96.1,277.3 98.8,277.1 101.6,277.0 104.3,276.8 107.1,276.5 109.8,276.3 112.5,276.0 115.3,275.6 118.0,275.2 120.8,274.8 123.5,274.3 126.2,273.7 129.0,273.1 131.7,272.4 134.4,271.6 137.2,270.8 139.9,269.9 142.6,268.9 145.4,267.8 148.1,266.6 150.9,265.3 153.6,264.0 156.3,262.5 159.1,261.0 161.8,259.3 164.6,257.6 167.3,255.8 170.0,253.8 172.8,251.8 175.5,249.8 178.2,247.6 181.0,245.4 183.7,243.1 186.4,240.8 189.2,238.4 191.9,236.0 194.7,233.5 197.4,231.1 200.1,228.6 202.9,226.1 205.6,223.6 208.3,221.2 211.1,218.8 213.8,216.4 216.6,214.1 219.3,211.9 222.0,209.7 224.8,207.6 227.5,205.6 230.2,203.7 233.0,201.9 235.7,200.3 238.5,198.8 241.2,197.4 243.9,196.1 246.7,195.0 249.4,194.0 252.2,193.2 254.9,192.5 257.6,192.0 260.4,191.6 263.1,191.4 265.8,191.3 268.6,191.4 271.3,191.7 274.0,192.0 276.8,192.5 279.5,193.2 282.3,194.0 285.0,194.8 287.7,195.9 290.5,197.0 293.2,198.2 296.0,199.5 298.7,201.0 301.4,202.5 304.2,204.0 306.9,205.7 309.6,207.4 312.4,209.1 315.1,210.9 317.9,212.8 320.6,214.6 323.3,216.5 326.1,218.4 328.8,220.4 331.5,222.3 334.3,224.2 337.0,226.1 339.8,228.1 342.5,230.0 345.2,231.8 348.0,233.7 350.7,235.5 353.4,237.3 356.2,239.1 358.9,240.8 361.7,242.5 364.4,244.1 367.1,245.7 369.9,247.3 372.6,248.8 375.3,250.3 378.1,251.7 380.8,253.0 383.6,254.4 386.3,255.6 389.0,256.8 391.8,258.0 394.5,259.1 397.2,260.2 400.0,261.2 402.7,262.2 405.4,263.2 408.2,264.0 410.9,264.9 413.7,265.7 416.4,266.5 419.1,267.2 421.9,267.9 424.6,268.5 427.3,269.2 430.1,269.7 432.8,270.3 435.6,270.8 438.3,271.3 441.0,271.8 443.8,272.2 446.5,272.6 449.2,273.0 452.0,273.3 454.7,273.7 457.5,274.0 460.2,274.3 462.9,274.5 465.7,274.8 468.4,275.0 471.2,275.3 473.9,275.5 476.6,275.7 479.4,275.8 482.1,276.0 484.8,276.2 487.6,276.3 490.3,276.4 493.1,276.6 495.8,276.7 498.5,276.8 501.3,276.9 504.0,277.0" fill="none" stroke="#6aa84f" stroke-width="2.2" stroke-linejoin="round"/>
<line x1="354" y1="30" x2="376" y2="30" stroke="#3d6ea5" stroke-width="2.6"/>
<text x="382" y="34" font-size="11.5" fill="currentColor">10 days</text>
<line x1="354" y1="48" x2="376" y2="48" stroke="#e69138" stroke-width="2.6"/>
<text x="382" y="52" font-size="11.5" fill="currentColor">1 month</text>
<line x1="354" y1="66" x2="376" y2="66" stroke="#6aa84f" stroke-width="2.6"/>
<text x="382" y="70" font-size="11.5" fill="currentColor">3 months</text>
<text x="285" y="320" font-size="12.5" fill="currentColor" text-anchor="middle">spot price S</text>
<text x="16" y="146" font-size="12.5" fill="currentColor" text-anchor="middle" transform="rotate(-90 16 146)">gamma</text>
</svg>
</p>

*Gamma of a call or put against spot for 10 days, 1 month and 3 months ($K=100$, $r=5\%$, $\sigma=25\%$): the
peak at the strike grows as $1/\sqrt\tau$ while the curve narrows.*

In [12]:
# ATM gamma grows like 1/sqrt(tau): check the scaling and show the narrowing spike
K, r, sigma = 100, 0.05, 0.25
rows = [("10 days", 10 / 365), ("1 month", 1 / 12), ("3 months", 0.25)]
base = greeks(K, K, r, sigma, 0.25)['gamma']
print("maturity   ATM gamma   ratio to 3-month   sqrt(0.25/tau)")
for lab, T in rows:
    g = greeks(K, K, r, sigma, T)['gamma']
    print(f"{lab:>9}   {g:8.4f}      {g / base:6.3f}            {math.sqrt(0.25 / T):6.3f}")
print("\ngamma at spot 90 / 95 / 100 / 105 / 110 (collapses away from the strike):")
for lab, T in rows:
    print(f"  {lab:>8}: " + "  ".join(f"{greeks(s, K, r, sigma, T)['gamma']:.4f}" for s in (90, 95, 100, 105, 110)))
print("\n0.4/(S sigma sqrt(tau)) estimate vs exact ATM gamma:",
      "  ".join(f"{0.4 / (K * sigma * math.sqrt(T)):.4f}/{greeks(K, K, r, sigma, T)['gamma']:.4f}" for _, T in rows))

maturity   ATM gamma   ratio to 3-month   sqrt(0.25/tau)
  10 days     0.0963       3.056             3.021
  1 month     0.0550       1.747             1.732
 3 months     0.0315       1.000             1.000

gamma at spot 90 / 95 / 100 / 105 / 110 (collapses away from the strike):
   10 days: 0.0048  0.0502  0.0963  0.0429  0.0054
   1 month: 0.0242  0.0481  0.0550  0.0391  0.0185
  3 months: 0.0281  0.0326  0.0315  0.0261  0.0189

0.4/(S sigma sqrt(tau)) estimate vs exact ATM gamma: 0.0967/0.0963  0.0554/0.0550  0.0320/0.0315


### 6.2.3 Theta

Theta is the change in option value as calendar time passes, $\Theta=\dfrac{\partial V}{\partial t}=-\dfrac{\partial V}{\partial\tau}$,
quoted per year (divide by $365$ for a per-day figure). **General formulas (continuous dividend yield $y$):**
$$\Theta_{\text{call}}=-\frac{S\,e^{-y\tau}N'(d_1)\,\sigma}{2\sqrt\tau}+y\,S\,e^{-y\tau}N(d_1)-r\,K\,e^{-r\tau}N(d_2),$$
$$\Theta_{\text{put}}=-\frac{S\,e^{-y\tau}N'(d_1)\,\sigma}{2\sqrt\tau}-y\,S\,e^{-y\tau}N(-d_1)+r\,K\,e^{-r\tau}N(-d_2).$$
Each has three pieces: the **optionality decay** (the first term, the same for call and put and always
negative), the **dividend carry** (the $y$ term) and the **interest on the strike** (the $r$ term). With no
dividends, $\Theta_{\text{call}}=-\dfrac{S\,N'(d_1)\sigma}{2\sqrt\tau}-rKe^{-r\tau}N(d_2)<0$: a call always loses value
as time passes.

<p align="center">
<svg viewBox="0 0 520 330" width="520" xmlns="http://www.w3.org/2000/svg" style="max-width:100%;height:auto" font-family="sans-serif">
<line x1="66" y1="278.0" x2="504" y2="278.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="282.0" font-size="11" fill="currentColor" text-anchor="end">-40</text>
<line x1="66" y1="212.0" x2="504" y2="212.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="216.0" font-size="11" fill="currentColor" text-anchor="end">-30</text>
<line x1="66" y1="146.0" x2="504" y2="146.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="150.0" font-size="11" fill="currentColor" text-anchor="end">-20</text>
<line x1="66" y1="80.0" x2="504" y2="80.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="84.0" font-size="11" fill="currentColor" text-anchor="end">-10</text>
<line x1="66" y1="14.0" x2="504" y2="14.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="18.0" font-size="11" fill="currentColor" text-anchor="end">0</text>
<line x1="66.0" y1="14" x2="66.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="66.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">60</text>
<line x1="175.5" y1="14" x2="175.5" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="175.5" y="294" font-size="11" fill="currentColor" text-anchor="middle">80</text>
<line x1="285.0" y1="14" x2="285.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="285.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">100</text>
<line x1="394.5" y1="14" x2="394.5" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="394.5" y="294" font-size="11" fill="currentColor" text-anchor="middle">120</text>
<line x1="504.0" y1="14" x2="504.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="504.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">140</text>
<rect x="66" y="14" width="438" height="264" fill="none" stroke="currentColor" stroke-opacity="0.45"/>
<line x1="285.0" y1="14" x2="285.0" y2="278" stroke="currentColor" stroke-opacity="0.5" stroke-dasharray="4 3"/>
<text x="289.0" y="26" font-size="11" fill="currentColor" fill-opacity="0.8">K = 100</text>
<polyline points="66.0,14.0 68.7,14.0 71.5,14.0 74.2,14.0 77.0,14.0 79.7,14.0 82.4,14.0 85.2,14.0 87.9,14.0 90.6,14.0 93.4,14.0 96.1,14.0 98.8,14.0 101.6,14.0 104.3,14.0 107.1,14.0 109.8,14.0 112.5,14.0 115.3,14.0 118.0,14.0 120.8,14.0 123.5,14.0 126.2,14.0 129.0,14.0 131.7,14.0 134.4,14.0 137.2,14.0 139.9,14.0 142.6,14.0 145.4,14.0 148.1,14.0 150.9,14.0 153.6,14.0 156.3,14.0 159.1,14.0 161.8,14.0 164.6,14.0 167.3,14.0 170.0,14.0 172.8,14.0 175.5,14.0 178.2,14.0 181.0,14.0 183.7,14.0 186.4,14.0 189.2,14.0 191.9,14.0 194.7,14.0 197.4,14.0 200.1,14.1 202.9,14.1 205.6,14.2 208.3,14.3 211.1,14.5 213.8,14.7 216.6,15.2 219.3,15.8 222.0,16.7 224.8,18.0 227.5,19.8 230.2,22.2 233.0,25.4 235.7,29.6 238.5,34.9 241.2,41.5 243.9,49.5 246.7,58.9 249.4,69.9 252.2,82.4 254.9,96.2 257.6,111.1 260.4,126.9 263.1,143.0 265.8,159.1 268.6,174.6 271.3,189.0 274.0,201.8 276.8,212.5 279.5,220.8 282.3,226.5 285.0,229.2 287.7,229.0 290.5,226.1 293.2,220.5 296.0,212.7 298.7,203.0 301.4,191.7 304.2,179.5 306.9,166.6 309.6,153.6 312.4,140.8 315.1,128.4 317.9,116.8 320.6,106.1 323.3,96.4 326.1,87.9 328.8,80.4 331.5,73.9 334.3,68.5 337.0,63.9 339.8,60.2 342.5,57.2 345.2,54.8 348.0,52.9 350.7,51.4 353.4,50.2 356.2,49.3 358.9,48.7 361.7,48.2 364.4,47.8 367.1,47.6 369.9,47.4 372.6,47.3 375.3,47.2 378.1,47.1 380.8,47.0 383.6,47.0 386.3,47.0 389.0,47.0 391.8,47.0 394.5,47.0 397.2,47.0 400.0,47.0 402.7,47.0 405.4,47.0 408.2,47.0 410.9,47.0 413.7,47.0 416.4,47.0 419.1,47.0 421.9,47.0 424.6,47.0 427.3,47.0 430.1,47.0 432.8,47.0 435.6,47.0 438.3,47.0 441.0,47.0 443.8,47.0 446.5,47.0 449.2,47.0 452.0,47.0 454.7,47.0 457.5,47.0 460.2,47.0 462.9,47.0 465.7,47.0 468.4,47.0 471.2,47.0 473.9,47.0 476.6,47.0 479.4,47.0 482.1,47.0 484.8,47.0 487.6,47.0 490.3,47.0 493.1,47.0 495.8,47.0 498.5,47.0 501.3,47.0 504.0,47.0" fill="none" stroke="#3d6ea5" stroke-width="2.2" stroke-linejoin="round"/>
<polyline points="66.0,14.0 68.7,14.0 71.5,14.0 74.2,14.0 77.0,14.0 79.7,14.0 82.4,14.0 85.2,14.0 87.9,14.0 90.6,14.0 93.4,14.0 96.1,14.0 98.8,14.0 101.6,14.0 104.3,14.0 107.1,14.0 109.8,14.0 112.5,14.0 115.3,14.0 118.0,14.0 120.8,14.0 123.5,14.0 126.2,14.0 129.0,14.0 131.7,14.0 134.4,14.0 137.2,14.0 139.9,14.0 142.6,14.0 145.4,14.0 148.1,14.0 150.9,14.1 153.6,14.1 156.3,14.1 159.1,14.2 161.8,14.2 164.6,14.3 167.3,14.5 170.0,14.6 172.8,14.8 175.5,15.1 178.2,15.4 181.0,15.8 183.7,16.3 186.4,16.9 189.2,17.6 191.9,18.5 194.7,19.5 197.4,20.7 200.1,22.2 202.9,23.9 205.6,25.8 208.3,28.0 211.1,30.5 213.8,33.3 216.6,36.4 219.3,39.9 222.0,43.7 224.8,47.7 227.5,52.1 230.2,56.8 233.0,61.8 235.7,67.0 238.5,72.4 241.2,77.9 243.9,83.6 246.7,89.3 249.4,95.0 252.2,100.7 254.9,106.2 257.6,111.6 260.4,116.7 263.1,121.6 265.8,126.0 268.6,130.1 271.3,133.7 274.0,136.9 276.8,139.5 279.5,141.6 282.3,143.2 285.0,144.2 287.7,144.7 290.5,144.6 293.2,144.0 296.0,142.9 298.7,141.4 301.4,139.4 304.2,137.0 306.9,134.2 309.6,131.2 312.4,127.9 315.1,124.4 317.9,120.7 320.6,116.9 323.3,113.0 326.1,109.1 328.8,105.1 331.5,101.2 334.3,97.4 337.0,93.7 339.8,90.0 342.5,86.5 345.2,83.2 348.0,80.0 350.7,77.0 353.4,74.2 356.2,71.5 358.9,69.1 361.7,66.8 364.4,64.7 367.1,62.7 369.9,61.0 372.6,59.3 375.3,57.9 378.1,56.6 380.8,55.4 383.6,54.3 386.3,53.4 389.0,52.5 391.8,51.8 394.5,51.1 397.2,50.5 400.0,50.0 402.7,49.6 405.4,49.2 408.2,48.8 410.9,48.5 413.7,48.3 416.4,48.1 419.1,47.9 421.9,47.7 424.6,47.6 427.3,47.5 430.1,47.4 432.8,47.3 435.6,47.2 438.3,47.2 441.0,47.1 443.8,47.1 446.5,47.0 449.2,47.0 452.0,47.0 454.7,47.0 457.5,46.9 460.2,46.9 462.9,46.9 465.7,46.9 468.4,46.9 471.2,46.9 473.9,46.9 476.6,46.9 479.4,46.9 482.1,46.9 484.8,46.9 487.6,46.9 490.3,46.9 493.1,46.9 495.8,46.9 498.5,46.9 501.3,46.9 504.0,46.9" fill="none" stroke="#e69138" stroke-width="2.2" stroke-linejoin="round"/>
<polyline points="66.0,14.0 68.7,14.0 71.5,14.0 74.2,14.0 77.0,14.1 79.7,14.1 82.4,14.1 85.2,14.1 87.9,14.1 90.6,14.2 93.4,14.2 96.1,14.3 98.8,14.3 101.6,14.4 104.3,14.5 107.1,14.6 109.8,14.7 112.5,14.8 115.3,14.9 118.0,15.1 120.8,15.3 123.5,15.5 126.2,15.8 129.0,16.1 131.7,16.4 134.4,16.8 137.2,17.2 139.9,17.7 142.6,18.2 145.4,18.8 148.1,19.4 150.9,20.1 153.6,20.8 156.3,21.7 159.1,22.5 161.8,23.5 164.6,24.5 167.3,25.6 170.0,26.8 172.8,28.1 175.5,29.4 178.2,30.8 181.0,32.3 183.7,33.9 186.4,35.5 189.2,37.2 191.9,39.0 194.7,40.8 197.4,42.7 200.1,44.7 202.9,46.7 205.6,48.7 208.3,50.8 211.1,52.9 213.8,55.0 216.6,57.1 219.3,59.3 222.0,61.5 224.8,63.6 227.5,65.7 230.2,67.9 233.0,69.9 235.7,72.0 238.5,74.0 241.2,75.9 243.9,77.8 246.7,79.7 249.4,81.4 252.2,83.1 254.9,84.7 257.6,86.2 260.4,87.6 263.1,88.9 265.8,90.2 268.6,91.3 271.3,92.3 274.0,93.2 276.8,94.0 279.5,94.7 282.3,95.3 285.0,95.7 287.7,96.1 290.5,96.4 293.2,96.5 296.0,96.6 298.7,96.6 301.4,96.4 304.2,96.2 306.9,95.9 309.6,95.5 312.4,95.0 315.1,94.5 317.9,93.8 320.6,93.2 323.3,92.4 326.1,91.6 328.8,90.7 331.5,89.8 334.3,88.9 337.0,87.9 339.8,86.9 342.5,85.8 345.2,84.8 348.0,83.7 350.7,82.6 353.4,81.5 356.2,80.4 358.9,79.2 361.7,78.1 364.4,77.0 367.1,75.9 369.9,74.8 372.6,73.7 375.3,72.6 378.1,71.6 380.8,70.5 383.6,69.5 386.3,68.5 389.0,67.5 391.8,66.6 394.5,65.6 397.2,64.7 400.0,63.8 402.7,63.0 405.4,62.2 408.2,61.4 410.9,60.6 413.7,59.9 416.4,59.2 419.1,58.5 421.9,57.8 424.6,57.2 427.3,56.6 430.1,56.0 432.8,55.5 435.6,55.0 438.3,54.5 441.0,54.0 443.8,53.5 446.5,53.1 449.2,52.7 452.0,52.3 454.7,51.9 457.5,51.6 460.2,51.3 462.9,51.0 465.7,50.7 468.4,50.4 471.2,50.1 473.9,49.9 476.6,49.7 479.4,49.5 482.1,49.3 484.8,49.1 487.6,48.9 490.3,48.7 493.1,48.6 495.8,48.4 498.5,48.3 501.3,48.2 504.0,48.1" fill="none" stroke="#6aa84f" stroke-width="2.2" stroke-linejoin="round"/>
<line x1="76" y1="216" x2="98" y2="216" stroke="#3d6ea5" stroke-width="2.6"/>
<text x="104" y="220" font-size="11.5" fill="currentColor">10 days</text>
<line x1="76" y1="234" x2="98" y2="234" stroke="#e69138" stroke-width="2.6"/>
<text x="104" y="238" font-size="11.5" fill="currentColor">1 month</text>
<line x1="76" y1="252" x2="98" y2="252" stroke="#6aa84f" stroke-width="2.6"/>
<text x="104" y="256" font-size="11.5" fill="currentColor">3 months</text>
<text x="285" y="320" font-size="12.5" fill="currentColor" text-anchor="middle">spot price S</text>
<text x="16" y="146" font-size="12.5" fill="currentColor" text-anchor="middle" transform="rotate(-90 16 146)">call theta (per year)</text>
</svg>
</p>

*Call theta against spot for 10 days, 1 month and 3 months ($K=100$, $r=5\%$, $\sigma=25\%$, per year). Theta is
most negative at the strike and deepens as expiry nears (the decay term scales like $1/\sqrt\tau$); far in the
money it settles at $-rKe^{-r\tau}\approx-5$ (the interest-on-strike term) and far out of the money at $0$.*

#### T1 — when does a European option have positive theta?

A long option normally bleeds value, but two situations make theta **positive** (for a European option).

**Solution 1 — a deep in-the-money put.** For $S\ll K$, $d_1,d_2\to-\infty$, so $N'(d_1)\to0$ (the decay term
vanishes) and $N(-d_2)\to1$:
$$\Theta_{\text{put}}\;\longrightarrow\;rKe^{-r\tau}>0 .$$
Intuition: a deep in-the-money put is certain to be exercised, so it behaves like a bond that pays $K-S_T$ at
maturity, worth $\approx Ke^{-r\tau}-S$. There is no optionality left to decay, while the present value of the
strike receipt **accretes interest** as maturity nears — the put's value *rises* with time. (An American put
would instead be exercised early here, which is why the effect is specific to European puts.)

**Solution 2 — an in-the-money call with a high dividend yield.** For $S\gg K$, $N(d_1),N(d_2)\to1$ and
$N'(d_1)\to0$:
$$\Theta_{\text{call}}\;\longrightarrow\;y\,S\,e^{-y\tau}-rK\,e^{-r\tau},$$
which is **positive when the dividend income $ySe^{-y\tau}$ exceeds the interest $rKe^{-r\tau}$ on the strike**
— roughly when $yS>rK$, e.g. a high yield on a stock well above the strike. Intuition: a deep in-the-money call is $\approx Se^{-y\tau}-Ke^{-r\tau}$. As time passes
the dividends the call-holder forgoes become fewer, so the stock leg *grows back* at rate $y$, while the strike's
present value grows at rate $r$; if $y$ is large the first effect wins.

In [13]:
# Theta in numbers: the graph values, and the two positive-theta cases
K, r, sigma = 100, 0.05, 0.25
print("call theta (per year) at spot 80 / 90 / 100 / 110 / 120   [per calendar day = /365]")
for lab, T in [("10 days", 10 / 365), ("1 month", 1 / 12), ("3 months", 0.25)]:
    print(f"  {lab:>8}: " + "  ".join(f"{greeks(s, K, r, sigma, T)['theta']:8.3f}" for s in (80, 90, 100, 110, 120)),
          f"   (ATM per day {greeks(K, K, r, sigma, T)['theta'] / 365:+.3f})")

print("\nSolution 1 - European put, K=100, r=5%, sigma=25%, tau=1:  theta vs spot (limit as S->0 is r K e^(-r tau))")
lim = r * K * math.exp(-r * 1.0)
for s in (10, 30, 50, 70, 90, 100, 120):
    print(f"   S={s:>3}: put theta = {greeks(s, K, r, sigma, 1.0, call=False)['theta']:+8.4f}")
print(f"   limit  r K e^(-r tau) = {lim:.4f}")

print("\nSolution 2 - European call, K=100, r=2%, y=10%, sigma=25%, tau=1: theta vs spot")
for s in (60, 100, 120, 150, 200, 300):
    th = greeks(s, K, 0.02, sigma, 1.0, call=True, y=0.10)['theta']
    lim2 = 0.10 * s * math.exp(-0.10) - 0.02 * K * math.exp(-0.02)
    print(f"   S={s:>3}: call theta = {th:+8.4f}   (y S e^(-y tau) - r K e^(-r tau) = {lim2:+8.4f}; matches only once S >> K)")

call theta (per year) at spot 80 / 90 / 100 / 110 / 120   [per calendar day = /365]
   10 days:   -0.000    -1.242   -32.605    -7.002    -4.995    (ATM per day -0.089)
   1 month:   -0.160    -6.489   -19.731   -11.520    -5.622    (ATM per day -0.054)
  3 months:   -2.338    -8.160   -12.386   -11.044    -7.822    (ATM per day -0.034)

Solution 1 - European put, K=100, r=5%, sigma=25%, tau=1:  theta vs spot (limit as S->0 is r K e^(-r tau))
   S= 10: put theta =  +4.7561
   S= 30: put theta =  +4.7561
   S= 50: put theta =  +4.6148
   S= 70: put theta =  +2.4338
   S= 90: put theta =  -1.4448
   S=100: put theta =  -2.4943
   S=120: put theta =  -2.4310
   limit  r K e^(-r tau) = 4.7561

Solution 2 - European call, K=100, r=2%, y=10%, sigma=25%, tau=1: theta vs spot
   S= 60: call theta =  -0.1653   (y S e^(-y tau) - r K e^(-r tau) =  +3.4686; matches only once S >> K)
   S=100: call theta =  -1.2459   (y S e^(-y tau) - r K e^(-r tau) =  +7.0880; matches only once S >> K)
   S=120: c

#### T2 — a delta-hedged long call when the stock suddenly moves (via the BSM equation)

**Set-up.** Go long a call on GM (no dividends) and short $\Delta$ shares: $\Pi=c-\Delta S$. The
Black–Scholes–Merton equation for the call, with $\Delta=\partial c/\partial S$, $\Gamma=\partial^2c/\partial S^2$, $\Theta=\partial c/\partial t$,
reads
$$\Theta+\tfrac12\sigma^{2}S^{2}\Gamma+rS\Delta-rc=0\quad\Longleftrightarrow\quad
\boxed{\ \Theta+\tfrac12\sigma^{2}S^{2}\Gamma=r\,\big(c-S\Delta\big)=r\,\Pi\ }.$$

**Step 1 — value change of the hedged portfolio over a short interval.** By the Taylor expansion of 6.2,
$$d\Pi=dc-\Delta\,dS=\Theta\,dt+\Delta\,dS+\tfrac12\Gamma\,(dS)^{2}-\Delta\,dS=\Theta\,dt+\tfrac12\Gamma\,(dS)^{2}.$$
The first-order term cancels: the portfolio is insensitive to small stock moves in *either* direction.

**Step 2 — an immediate change in the stock price.** If GM jumps by $\Delta S$ with essentially no time elapsed
($dt\approx0$),
$$\Delta\Pi\approx\tfrac12\,\Gamma\,(\Delta S)^{2}>0\qquad\text{whether }\Delta S\text{ is positive or negative,}$$
since $\Gamma>0$. The portfolio **gains** from any sufficiently large move — it is *long gamma*.

**Step 3 — is this an arbitrage? No.** The BSM equation says the gamma gain is paid for by time decay:
$$\Theta=r\,\Pi-\tfrac12\sigma^{2}S^{2}\Gamma .$$
Over time $dt$ the portfolio loses $|\Theta|\,dt$ to theta, while a stock with volatility $\sigma$ moves by
$(dS)^2\approx\sigma^2S^2\,dt$ on average, delivering an expected gamma gain of $\tfrac12\Gamma\sigma^2S^2\,dt$. Adding the
two, $E[d\Pi]=\Theta\,dt+\tfrac12\Gamma\sigma^2S^2\,dt=r\,\Pi\,dt$: **on average the hedged portfolio earns just the
risk-free rate**, exactly what no-arbitrage demands. The jump in GM gives a one-off gain, but the position has
been paying theta every day to earn that right. You profit only if the *realised* volatility exceeds the $\sigma$
implied in the option's price (and lose if it is lower) — a bet on volatility, not a free lunch.

In [14]:
# Delta-hedged long call: immediate jump in S gives about 0.5*Gamma*dS^2, and the BSM equation links Theta and Gamma
S0, K, r, sigma, T = 100, 100, 0.05, 0.25, 0.25
c0 = bs(S0, K, r, sigma, T); g = greeks(S0, K, r, sigma, T)
print(f"call {c0:.4f}, delta {g['delta']:.4f}, gamma {g['gamma']:.5f}, theta {g['theta']:.4f} per year")
print("\nimmediate jump dS (no time elapsed): change in hedged portfolio  vs  0.5*Gamma*dS^2")
for dS in (-10, -5, -2, 2, 5, 10):
    dPi = (bs(S0 + dS, K, r, sigma, T) - c0) - g['delta'] * dS
    print(f"   dS = {dS:+3d}:  change {dPi:+.4f}    0.5*Gamma*dS^2 = {0.5 * g['gamma'] * dS * dS:+.4f}")
Pi = c0 - S0 * g['delta']
print(f"\nBSM equation:  Theta + 0.5 sigma^2 S^2 Gamma = {g['theta'] + 0.5 * sigma ** 2 * S0 ** 2 * g['gamma']:+.6f}"
      f"   vs   r * Pi = {r * Pi:+.6f}   (equal -> no arbitrage)")
print(f"per day: theta bleeds {g['theta'] / 365:+.4f}, expected gamma gain at the implied vol {0.5 * sigma ** 2 * S0 ** 2 * g['gamma'] / 365:+.4f}; "
      f"their sum {(g['theta'] + 0.5 * sigma ** 2 * S0 ** 2 * g['gamma']) / 365:+.4f} is just the riskless growth r*Pi/365 = {r * Pi / 365:+.4f}")

call 5.5984, delta 0.5645, gamma 0.03150, theta -12.3855 per year

immediate jump dS (no time elapsed): change in hedged portfolio  vs  0.5*Gamma*dS^2
   dS = -10:  change +1.6101    0.5*Gamma*dS^2 = +1.5748
   dS =  -5:  change +0.4037    0.5*Gamma*dS^2 = +0.3937
   dS =  -2:  change +0.0638    0.5*Gamma*dS^2 = +0.0630
   dS =  +2:  change +0.0619    0.5*Gamma*dS^2 = +0.0630
   dS =  +5:  change +0.3745    0.5*Gamma*dS^2 = +0.3937
   dS = +10:  change +1.3976    0.5*Gamma*dS^2 = +1.5748

BSM equation:  Theta + 0.5 sigma^2 S^2 Gamma = -2.542800   vs   r * Pi = -2.542800   (equal -> no arbitrage)
per day: theta bleeds -0.0339, expected gamma gain at the implied vol +0.0270; their sum -0.0070 is just the riskless growth r*Pi/365 = -0.0070


### 6.2.4 Vega

**General formula (continuous dividend yield $y$).** Vega is the sensitivity of the price to volatility,
$$\text{Vega}=\frac{\partial V}{\partial\sigma}=S\,e^{-y\tau}\,N'(d_1)\,\sqrt\tau\;>0,$$
the **same for a call and a put** (their prices differ by a term that does not involve $\sigma$). Quoted per $1.00$
of volatility; divide by $100$ for the price change per one volatility *point*.

**Vega and time.** Vega grows with time to maturity, roughly like $\sqrt\tau$ (at the money
$\text{Vega}\approx0.4\,S\sqrt\tau$): more time means more room for volatility to matter, so **long-dated options
are far more sensitive to volatility** than short-dated ones (which are dominated instead by gamma and theta).

**Vega and the state of the option.** Vega is a **bell-shaped function of the spot**, peaking when the option is at
the money ($d_1\approx0$, where $N'(d_1)$ is largest) and decaying towards $0$ when the option is deep in or out
of the money ($N'(d_1)\to0$): a deep ITM or OTM option's fate is nearly settled, so volatility hardly matters.
ATM $\;>\;$ slightly ITM/OTM $\;>\;$ deep ITM/OTM.

In [15]:
# Vega vs spot (state of the option) and vs maturity; call and put are identical
K, r, sigma = 100, 0.05, 0.25
spots = (60, 80, 90, 100, 110, 120, 140)
print("spot:              " + "".join(f"{s:>8}" for s in spots))
for lab, T in [("3 months", 0.25), ("1 year", 1.0)]:
    print(f"vega, {lab:>8}:  " + "".join(f"{greeks(s, K, r, sigma, T)['vega']:8.2f}" for s in spots))
print("\nATM vega vs maturity (grows like sqrt(tau); estimate 0.4*S*sqrt(tau)):")
for lab, T in [("10 days", 10 / 365), ("1 month", 1 / 12), ("3 months", 0.25), ("1 year", 1.0)]:
    print(f"  {lab:>8}: vega {greeks(K, K, r, sigma, T)['vega']:7.3f}   estimate {0.4 * K * math.sqrt(T):7.3f}")
print("call vega == put vega:", abs(greeks(100, 95, r, sigma, 0.5)['vega'] - greeks(100, 95, r, sigma, 0.5, call=False)['vega']) < 1e-12)

spot:                    60      80      90     100     110     120     140
vega, 3 months:      0.01    4.28   14.24   19.69   14.30    6.43    0.48
vega,   1 year:      5.47   27.17   35.74   37.84   34.20   27.46   13.83

ATM vega vs maturity (grows like sqrt(tau); estimate 0.4*S*sqrt(tau)):
   10 days: vega   6.594   estimate   6.621
   1 month: vega  11.466   estimate  11.547
  3 months: vega  19.685   estimate  20.000
    1 year: vega  37.842   estimate  40.000
call vega == put vega: True


#### V1 — implied volatility and the volatility smile

**Implied volatility.** The Black–Scholes price depends on a single number we cannot observe, the volatility
$\sigma$. The **implied volatility** $\sigma_{\text{imp}}$ is the value that makes the Black–Scholes price equal the
**market** price:
$$c_{\text{BS}}(S,K,r,\tau,\sigma_{\text{imp}})=c_{\text{market}} .$$
Because Vega $>0$, the price is strictly increasing in $\sigma$, so the solution is **unique**; it has no closed
form and is found numerically (bisection or Newton, using Vega as the derivative). Quoting an option by its
implied volatility is simply a convenient, price-equivalent unit.

**Volatility smile.** If Black–Scholes were literally true, the implied volatility would be the **same for every
strike** (and maturity). In real markets, plotting $\sigma_{\text{imp}}$ against $K$ gives a **curve**: a *smile*
(U-shape, typical in FX) or a *skew/smirk* (typically higher for low strikes in equities). The figure shows a
smile produced by a market whose volatility is uncertain (equally likely $20\%$ or $40\%$, mean $30\%$):
Black–Scholes prices at a flat $30\%$ would be wrong in the wings.

<p align="center">
<svg viewBox="0 0 520 330" width="520" xmlns="http://www.w3.org/2000/svg" style="max-width:100%;height:auto" font-family="sans-serif">
<line x1="66" y1="278.0" x2="504" y2="278.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="282.0" font-size="11" fill="currentColor" text-anchor="end">0.290</text>
<line x1="66" y1="234.0" x2="504" y2="234.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="238.0" font-size="11" fill="currentColor" text-anchor="end">0.300</text>
<line x1="66" y1="190.0" x2="504" y2="190.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="194.0" font-size="11" fill="currentColor" text-anchor="end">0.310</text>
<line x1="66" y1="146.0" x2="504" y2="146.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="150.0" font-size="11" fill="currentColor" text-anchor="end">0.320</text>
<line x1="66" y1="102.0" x2="504" y2="102.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="106.0" font-size="11" fill="currentColor" text-anchor="end">0.330</text>
<line x1="66" y1="58.0" x2="504" y2="58.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="62.0" font-size="11" fill="currentColor" text-anchor="end">0.340</text>
<line x1="66" y1="14.0" x2="504" y2="14.0" stroke="currentColor" stroke-opacity="0.13"/>
<text x="60" y="18.0" font-size="11" fill="currentColor" text-anchor="end">0.350</text>
<line x1="66.0" y1="14" x2="66.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="66.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">60</text>
<line x1="175.5" y1="14" x2="175.5" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="175.5" y="294" font-size="11" fill="currentColor" text-anchor="middle">80</text>
<line x1="285.0" y1="14" x2="285.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="285.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">100</text>
<line x1="394.5" y1="14" x2="394.5" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="394.5" y="294" font-size="11" fill="currentColor" text-anchor="middle">120</text>
<line x1="504.0" y1="14" x2="504.0" y2="278" stroke="currentColor" stroke-opacity="0.07"/>
<text x="504.0" y="294" font-size="11" fill="currentColor" text-anchor="middle">140</text>
<rect x="66" y="14" width="438" height="264" fill="none" stroke="currentColor" stroke-opacity="0.45"/>
<line x1="66" y1="234.0" x2="504" y2="234.0" stroke="currentColor" stroke-opacity="0.55" stroke-dasharray="6 4"/>
<text x="74" y="249.0" font-size="11" fill="currentColor" fill-opacity="0.85">flat 30% (Black-Scholes)</text>
<polyline points="66.0,44.1 77.0,57.8 87.9,71.7 98.8,85.5 109.8,99.3 120.8,112.8 131.7,126.0 142.6,138.7 153.6,150.8 164.6,162.3 175.5,173.1 186.4,183.0 197.4,192.1 208.3,200.3 219.3,207.7 230.2,214.1 241.2,219.6 252.2,224.3 263.1,228.1 274.0,231.2 285.0,233.4 296.0,234.8 306.9,235.5 317.9,235.6 328.8,235.0 339.8,233.8 350.7,232.0 361.7,229.7 372.6,226.9 383.6,223.6 394.5,219.9 405.4,215.8 416.4,211.4 427.3,206.7 438.3,201.6 449.2,196.4 460.2,190.9 471.2,185.2 482.1,179.4 493.1,173.5 504.0,167.5" fill="none" stroke="#3d6ea5" stroke-width="2.2" stroke-linejoin="round"/>
<line x1="354" y1="30" x2="376" y2="30" stroke="#3d6ea5" stroke-width="2.6"/>
<text x="382" y="34" font-size="11.5" fill="currentColor">implied vol</text>
<text x="285" y="320" font-size="12.5" fill="currentColor" text-anchor="middle">strike K</text>
<text x="16" y="146" font-size="12.5" fill="currentColor" text-anchor="middle" transform="rotate(-90 16 146)">implied volatility</text>
</svg>
</p>

**Implications for Black–Scholes pricing.**
1. **The constant-volatility, log-normal assumption is violated.** A single $\sigma$ cannot price all strikes: the
   market assigns more probability to large moves (fatter tails) than a log-normal distribution does — equity
   markets additionally fear crashes, giving a heavier left tail — so out-of-the-money options cost more than flat-vol
   Black–Scholes says (made precise by Breeden–Litzenberger, V3).
2. **Black–Scholes survives as a quoting language, not a literal model.** Traders quote prices as implied
   volatilities and build a whole **implied-volatility surface** (strike $\times$ maturity); the formula is the
   map between price and volatility.
3. **Other models are needed.** To price consistently across strikes and exotic payoffs one calibrates to the
   smile with local-volatility (Dupire), stochastic-volatility (Heston) or jump models.
4. **Hedge ratios depend on the smile.** The Black–Scholes delta and vega use a single $\sigma$, so how implied
   volatility moves when the stock moves (the smile dynamics) changes the true hedge.

In [16]:
def implied_vol(price, S, K, r, T, call=True):
    """Solve bs(sigma) = price by bisection (the price is strictly increasing in sigma because vega > 0)."""
    lo, hi = 1e-4, 3.0
    for _ in range(80):
        mid = (lo + hi) / 2
        if bs(S, K, r, mid, T, call=call) < price:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

# round trip: recover a known volatility from its own price
print("round trip: sigma = 0.2731 ->", round(implied_vol(bs(100, 110, 0.05, 0.2731, 1.0), 100, 110, 0.05, 1.0), 6))

# a market where volatility is uncertain: sigma = 20% or 40% with equal probability (mean 30%)
S, r, T = 100, 0.05, 1.0
print("\nstrike   mixture price   Black-Scholes at flat 30%   implied vol")
for K in (60, 70, 80, 90, 100, 110, 120, 130, 140):
    p = 0.5 * bs(S, K, r, 0.2, T) + 0.5 * bs(S, K, r, 0.4, T)
    print(f"{K:>5}   {p:12.4f}   {bs(S, K, r, 0.3, T):20.4f}   {implied_vol(p, S, K, r, T):10.4f}")

round trip: sigma = 0.2731 -> 0.2731

strike   mixture price   Black-Scholes at flat 30%   implied vol
   60        43.4757                43.1950       0.3432
   70        34.7774                34.3953       0.3275
   80        26.7826                26.4621       0.3138
   90        19.8421                19.6974       0.3045
  100        14.2368                14.2313       0.3001
  110        10.0222                10.0201       0.3001
  120         7.0267                 6.9040       0.3032
  130         4.9673                 4.6734       0.3086
  140         3.5644                 3.1187       0.3151


#### V2 — constant 30% volatility versus random volatility with mean 30% (Volga)

**Question.** Is a European call cheaper with a constant $30\%$ volatility or when the volatility is drawn from a
random distribution with mean $30\%$?

**Volga (vomma)** is the second derivative of the price in volatility, $\text{Volga}=\dfrac{\partial^{2}V}{\partial\sigma^{2}}=\dfrac{\partial\,\text{Vega}}{\partial\sigma}$ —
the convexity of the price in $\sigma$. We derive it from Vega.

**Step 1 — start from Vega.** $\text{Vega}=S\sqrt\tau\,N'(d_1)$ (no dividends).

**Step 2 — differentiate in $\sigma$.** Only $d_1$ depends on $\sigma$, and $N''(x)=-x\,N'(x)$:
$$\text{Volga}=S\sqrt\tau\,N''(d_1)\,\frac{\partial d_1}{\partial\sigma}=-S\sqrt\tau\,d_1\,N'(d_1)\,\frac{\partial d_1}{\partial\sigma}.$$

**Step 3 — $\partial d_1/\partial\sigma$.** From $d_1=\dfrac{\ln(S/K)+(r+\sigma^{2}/2)\tau}{\sigma\sqrt\tau}$,
$$\frac{\partial d_1}{\partial\sigma}=\frac{\sigma\tau}{\sigma\sqrt\tau}-\frac{d_1}{\sigma}=\sqrt\tau-\frac{d_1}{\sigma}=-\frac{d_1-\sigma\sqrt\tau}{\sigma}=-\frac{d_2}{\sigma}.$$

**Step 4 — combine.**
$$\boxed{\ \text{Volga}=S\sqrt\tau\,N'(d_1)\,\frac{d_1\,d_2}{\sigma}=\text{Vega}\cdot\frac{d_1\,d_2}{\sigma}\ }.$$

**Step 5 — apply to random volatility.** Let $\tilde\sigma$ be random with mean $\bar\sigma=30\%$ and variance
$\operatorname{Var}(\tilde\sigma)$. A second-order Taylor expansion of the price around $\bar\sigma$ (Jensen's
inequality in action) gives
$$E\big[c(\tilde\sigma)\big]\;\approx\;c(\bar\sigma)+\tfrac12\,\text{Volga}\cdot\operatorname{Var}(\tilde\sigma).$$
So the random-volatility option is **more expensive** exactly where $\text{Volga}>0$, i.e. where $d_1d_2>0$, and
cheaper where $d_1d_2<0$.

**Where is that?** $d_1d_2<0$ only when $0<d_1<\sigma\sqrt\tau$ (i.e. $d_2<0<d_1$), which works out to a narrow band
of strikes right around the forward price,
$$S\,e^{(r-\sigma^{2}/2)\tau}\;<\;K\;<\;S\,e^{(r+\sigma^{2}/2)\tau}.$$
For $S=100$, $r=5\%$, $\tau=1$, $\bar\sigma=30\%$ this is $K\in(100.5,\ 110.0)$. **Everywhere else — all the
in-the-money and out-of-the-money wings — $\text{Volga}>0$.**

**Answer.** Because the price is *convex* in volatility away from the money, uncertainty about volatility
**raises** the price: the random-volatility call is **more expensive** for in-the-money and out-of-the-money
strikes, with the largest premium in the wings (this is what generates the smile of V1). Only in the thin band of
strikes near the forward is Volga slightly negative, where randomising volatility makes the call marginally
*cheaper*; exactly at the money the two are almost equal.

In [17]:
# Volga = Vega * d1 d2 / sigma  vs finite difference; then constant 30% vs random volatility (20% or 40%, mean 30%)
S, r, T, sbar = 100, 0.05, 1.0, 0.30
var = 0.5 * (0.2 - sbar) ** 2 + 0.5 * (0.4 - sbar) ** 2          # Var(sigma~) = 0.01

def volga(S, K, r, sigma, T):
    d1, d2 = d1_d2(S, K, r, sigma, T)
    return greeks(S, K, r, sigma, T)['vega'] * d1 * d2 / sigma

h = 1e-3
print("strike   Volga formula   finite diff   | c(30%)    E[c(sigma~)]   difference   0.5*Volga*Var")
for K in (70, 80, 90, 100, 105, 110, 120, 130):
    fd = (bs(S, K, r, sbar + h, T) - 2 * bs(S, K, r, sbar, T) + bs(S, K, r, sbar - h, T)) / h ** 2
    c_const = bs(S, K, r, sbar, T)
    c_rand = 0.5 * bs(S, K, r, 0.2, T) + 0.5 * bs(S, K, r, 0.4, T)
    print(f"{K:>5}   {volga(S, K, r, sbar, T):12.4f}   {fd:11.4f}   | {c_const:7.4f}   {c_rand:10.4f}   {c_rand - c_const:+10.4f}   "
          f"{0.5 * volga(S, K, r, sbar, T) * var:+10.4f}")

k_hi = S * math.exp((r + sbar ** 2 / 2) * T)          # d1 = 0
k_lo = S * math.exp((r - sbar ** 2 / 2) * T)          # d1 = sigma sqrt(tau)  (i.e. d2 = 0)
print(f"\nVolga < 0 only for strikes between {k_lo:.1f} and {k_hi:.1f} (d1 d2 < 0): there random vol is slightly CHEAPER;")
print("everywhere else (ITM and OTM wings) Volga > 0, so random volatility makes the call MORE expensive.")

strike   Volga formula   finite diff   | c(30%)    E[c(sigma~)]   difference   0.5*Volga*Var
   70        77.7080       77.7079   | 34.3953      34.7774      +0.3821      +0.3885
   80        61.1182       61.1185   | 26.4621      26.7826      +0.3205      +0.3056
   90        26.1405       26.1408   | 19.6974      19.8421      +0.1447      +0.1307
  100         0.6675        0.6676   | 14.2313      14.2368      +0.0055      +0.0033
  105        -2.9546       -2.9546   | 11.9769      11.9621      -0.0148      -0.0148
  110         0.0414        0.0414   | 10.0201      10.0222      +0.0021      +0.0002
  120        21.9297       21.9299   |  6.9040       7.0267      +0.1227      +0.1096
  130        54.4721       54.4725   |  4.6734       4.9673      +0.2939      +0.2724

Volga < 0 only for strikes between 100.5 and 110.0 (d1 d2 < 0): there random vol is slightly CHEAPER;
everywhere else (ITM and OTM wings) Volga > 0, so random volatility makes the call MORE expensive.


#### V3 — recovering the risk-neutral density from call prices (Breeden–Litzenberger)

**Question.** Black–Scholes assumes the stock follows geometric Brownian motion. Suppose we do not know the
process, but we observe European call prices $c(K)$ for **every** strike $K$ with the same maturity $T$. Find the
risk-neutral density $f(s)$ of $S_T$.

**Step 1 — write the call as an integral against the unknown density.** Under the risk-neutral measure,
$$c(K)=e^{-r\tau}\,E^{Q}\big[(S_T-K)^{+}\big]=e^{-r\tau}\int_{K}^{\infty}(s-K)\,f(s)\,ds .$$

**Step 2 — first derivative in $K$ (Leibniz rule).** $K$ appears in the lower limit and in the integrand:
$$\frac{\partial c}{\partial K}=e^{-r\tau}\Big[-(K-K)f(K)+\int_{K}^{\infty}(-1)\,f(s)\,ds\Big]=-e^{-r\tau}\int_{K}^{\infty}f(s)\,ds=-e^{-r\tau}\,P^{Q}(S_T>K).$$
The boundary term vanishes because the integrand $(s-K)$ is zero at $s=K$. (So the slope of $c$ in $K$ gives the
risk-neutral exercise probability.)

**Step 3 — second derivative in $K$.** Differentiate the integral $\int_K^\infty f(s)\,ds$ once more; its derivative in
its lower limit is $-f(K)$:
$$\frac{\partial^{2}c}{\partial K^{2}}=-e^{-r\tau}\big(-f(K)\big)=e^{-r\tau}f(K).$$

**Step 4 — the result.**
$$\boxed{\ f(K)=e^{\,r\tau}\,\frac{\partial^{2}c}{\partial K^{2}}\ }$$
The risk-neutral density at $K$ is the (future-valued) **curvature of the call-price curve** — no assumption about the
stock's process is needed. In practice, three strikes $K-h,\,K,\,K+h$ give a **butterfly spread** (long one call at
$K-h$ and at $K+h$, short two at $K$) whose price $c(K-h)-2c(K)+c(K+h)\approx h^{2}e^{-r\tau}f(K)$, so
$f(K)\approx e^{r\tau}\dfrac{c(K-h)-2c(K)+c(K+h)}{h^{2}}$: a butterfly is a (discounted) bet on the stock landing near $K$.
Convexity of $c$ in $K$ ($f\ge0$) is exactly the no-arbitrage condition on call prices.

**Check against Black–Scholes.** Differentiating the Black–Scholes call in $K$ (the Delta identity of D1, Step 4,
cancels all but one term) gives $\partial c/\partial K=-e^{-r\tau}N(d_2)$, and again
$\partial^2c/\partial K^2=e^{-r\tau}\dfrac{N'(d_2)}{K\sigma\sqrt\tau}$ (since $\partial d_2/\partial K=-\dfrac{1}{K\sigma\sqrt\tau}$). Hence
$$f(K)=\frac{N'(d_2)}{K\,\sigma\sqrt\tau}=\frac{1}{K\sigma\sqrt{2\pi\tau}}\exp\!\Big[-\frac{\big(\ln K-\ln S-(r-\sigma^{2}/2)\tau\big)^{2}}{2\sigma^{2}\tau}\Big],$$
the **log-normal density** — Breeden–Litzenberger recovers it from the prices alone.

In [18]:
import math

def lognormal_pdf(s, S, r, sigma, T):
    m = math.log(S) + (r - sigma ** 2 / 2) * T; v = sigma * math.sqrt(T)
    return math.exp(-(math.log(s) - m) ** 2 / (2 * v * v)) / (s * v * math.sqrt(2 * math.pi))

def risk_neutral_density(call_price, K, r, T, h=0.5):
    """Breeden-Litzenberger: f(K) = e^{rT} d2c/dK2, estimated from three strikes (a butterfly spread)."""
    return math.exp(r * T) * (call_price(K + h) - 2 * call_price(K) + call_price(K - h)) / h ** 2

def exercise_prob(call_price, K, r, T, h=0.5):
    """P(S_T > K) = -e^{rT} dc/dK."""
    return -math.exp(r * T) * (call_price(K + h) - call_price(K - h)) / (2 * h)

S, r, T, sig = 100, 0.05, 1.0, 0.30
c_bs = lambda K: bs(S, K, r, sig, T)                                    # price list that happens to be Black-Scholes
print("Black-Scholes prices: density from call prices alone vs the log-normal formula")
for K in (70, 90, 100, 120, 150):
    print(f"   K={K:>3}:  Breeden-Litzenberger {risk_neutral_density(c_bs, K, r, T):.6f}    log-normal {lognormal_pdf(K, S, r, sig, T):.6f}")

# integrate the extracted density: total mass 1 and mean S e^{rT}
mass = mean = 0.0
for i in range(4, 1400):
    K = i * 0.5; f = risk_neutral_density(c_bs, K, r, T); mass += f * 0.5; mean += K * f * 0.5
print(f"\nintegral of f = {mass:.6f}   mean of f = {mean:.4f}   (S e^(rT) = {S * math.exp(r * T):.4f})")

# an unknown (non log-normal) market: sigma = 20% or 40% with equal probability -> fatter tails than a flat 30%
c_mix = lambda K: 0.5 * bs(S, K, r, 0.2, T) + 0.5 * bs(S, K, r, 0.4, T)
print("\nmixture market (not log-normal): density = the 50/50 mix of the two log-normals, from prices only")
for K in (60, 100, 160, 220):
    exact = 0.5 * lognormal_pdf(K, S, r, 0.2, T) + 0.5 * lognormal_pdf(K, S, r, 0.4, T)
    print(f"   K={K:>3}:  Breeden-Litzenberger {risk_neutral_density(c_mix, K, r, T):.6f}    mix of log-normals {exact:.6f}")
print(f"P(S_T > 220):  mixture market {exercise_prob(c_mix, 220, r, T):.5f}   vs flat-30% log-normal {exercise_prob(c_bs, 220, r, T):.5f}"
      "   -> the smile prices fatter tails")

Black-Scholes prices: density from call prices alone vs the log-normal formula
   K= 70:  Breeden-Litzenberger 0.009185    log-normal 0.009185
   K= 90:  Breeden-Litzenberger 0.013809    log-normal 0.013809
   K=100:  Breeden-Litzenberger 0.013296    log-normal 0.013296
   K=120:  Breeden-Litzenberger 0.009306    log-normal 0.009306
   K=150:  Breeden-Litzenberger 0.003637    log-normal 0.003637

integral of f = 1.000000   mean of f = 105.1271   (S e^(rT) = 105.1271)

mixture market (not log-normal): density = the 50/50 mix of the two log-normals, from prices only
   K= 60:  Breeden-Litzenberger 0.004465    mix of log-normals 0.004465
   K=100:  Breeden-Litzenberger 0.014834    mix of log-normals 0.014835
   K=160:  Breeden-Litzenberger 0.001981    mix of log-normals 0.001981
   K=220:  Breeden-Litzenberger 0.000283    mix of log-normals 0.000283
P(S_T > 220):  mixture market 0.01022   vs flat-30% log-normal 0.00451   -> the smile prices fatter tails


## 6.3 Option Portfolios and Exotic Options

Options are **building blocks**. Combine calls, puts and the stock itself and you can sculpt the payoff to match
almost any view on the market; change the rules of the payoff itself and you get an **exotic** option. The
notation of §6.1 carries over: $S_T$ is the stock price at maturity $T$, $K$ a strike, $\tau=T-t$, $r$ the
risk-free rate, $\sigma$ the volatility, and $c,\ p$ the European call and put prices. Every strategy below has two
sides: a **premium** paid or received today, and a **payoff** at maturity.

| strategy | construction | payoff at $T$ | in plain terms |
|---|---|---|---|
| **Covered call** | long 1 share, short 1 call (strike $K$) | $S_T-(S_T-K)^+=\min(S_T,K)$ | *Rent out your upside.* You keep the stock and pocket the call premium now, but give up every gain above $K$. Suits a flat or gently rising market. |
| **Protective put** | long 1 share, long 1 put (strike $K$) | $S_T+(K-S_T)^+=\max(S_T,K)$ | *Insurance with a deductible.* Your wealth can never fall below $K$; the put premium is the insurance bill. |
| **Bull spread** (calls) | long call $K_1$, short call $K_2$, with $K_1<K_2$ | $(S_T-K_1)^+-(S_T-K_2)^+$, a ramp from $0$ up to $K_2-K_1$ | *A cheaper, capped bet on a rise.* Selling the higher call pays part of the cost of buying the lower one, at the price of giving up gains above $K_2$. |
| **Bear spread** (puts) | long put $K_2$, short put $K_1$, with $K_1<K_2$ | $(K_2-S_T)^+-(K_1-S_T)^+$, a ramp from $K_2-K_1$ down to $0$ | The mirror image: a cheaper, capped bet on a *fall*. |
| **Butterfly** | long call $K_1$, short 2 calls $K_2=\tfrac{K_1+K_3}{2}$, long call $K_3$ | a tent: $0$ below $K_1$, peak $K_2-K_1$ at $K_2$, $0$ above $K_3$ | *A bet that the stock stays put.* Cheap, with a small known maximum loss (the premium) and the best payoff if $S_T$ finishes at $K_2$. |
| **Straddle** | long call and long put, same strike $K$ | $\lvert S_T-K\rvert$ | *A bet on a big move, direction unknown.* You profit if the stock moves far enough either way. |
| **Binary** (cash-or-nothing call) | pays a fixed cash amount $Q$ if the stock ends above $K$ | $Q\cdot\mathbf 1_{S_T>K}$ | *An all-or-nothing lottery ticket* — the payoff jumps from $0$ to $Q$ at the strike. |
| **Barrier** | a vanilla option that switches **off** (knock-out) or **on** (knock-in) if the price touches a barrier $H$ before $T$ | e.g. down-and-out call: $(S_T-K)^+\cdot\mathbf 1_{\min_{t\le T}S_t>H}$ | *An option with a tripwire.* A knock-out dies if the tripwire is touched, so it is cheaper than the vanilla; a knock-in only springs to life if it is touched. Knock-in $+$ knock-out $=$ vanilla. |
| **Asian** | payoff depends on the **average** price over the period | average-price call $(\bar S-K)^+$ with $\bar S=\tfrac1n\sum_i S_{t_i}$ | *A smoothed option.* Averaging damps volatility (a continuous arithmetic average has variance about $\tfrac13$ of the final price), so it is cheaper than a vanilla and hard to manipulate at expiry. |
| **Chooser** | at a fixed date $t_1<T$ the holder chooses whether it becomes a call or a put (same $K$, $T$) | $\max(c,p)$ at time $t_1$ | *Buy now, decide later* whether you wanted to bet up or down. Always dearer than either vanilla alone. |

<p align="center">
<svg viewBox="0 0 1000 432" width="1000" xmlns="http://www.w3.org/2000/svg" style="max-width:100%;height:auto" font-family="sans-serif">
<rect x="10" y="26" width="230" height="150" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<line x1="125.0" y1="26" x2="125.0" y2="176" stroke="currentColor" stroke-opacity="0.12"/>
<text x="125.0" y="190" font-size="11" fill="currentColor" text-anchor="middle">K</text>
<polyline points="10.0,167.7 240.0,34.3" fill="none" stroke="currentColor" stroke-opacity="0.45" stroke-width="1.4" stroke-dasharray="5 3"/>
<polyline points="10.0,167.7 125.0,101.0 240.0,101.0" fill="none" stroke="#3d6ea5" stroke-width="2.6" stroke-linejoin="round"/>
<text x="125" y="16" font-size="12.5" font-weight="bold" fill="currentColor" text-anchor="middle">Covered call</text>
<text x="14" y="39" font-size="10.5" fill="currentColor" fill-opacity="0.75">stock + short call</text>
<rect x="260" y="26" width="230" height="150" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<line x1="375.0" y1="26" x2="375.0" y2="176" stroke="currentColor" stroke-opacity="0.12"/>
<text x="375.0" y="190" font-size="11" fill="currentColor" text-anchor="middle">K</text>
<polyline points="260.0,167.7 490.0,34.3" fill="none" stroke="currentColor" stroke-opacity="0.45" stroke-width="1.4" stroke-dasharray="5 3"/>
<polyline points="260.0,101.0 375.0,101.0 490.0,34.3" fill="none" stroke="#3d6ea5" stroke-width="2.6" stroke-linejoin="round"/>
<text x="375" y="16" font-size="12.5" font-weight="bold" fill="currentColor" text-anchor="middle">Protective put</text>
<text x="264" y="39" font-size="10.5" fill="currentColor" fill-opacity="0.75">stock + long put</text>
<rect x="510" y="26" width="230" height="150" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<line x1="510" y1="154.6" x2="740" y2="154.6" stroke="currentColor" stroke-opacity="0.35" stroke-dasharray="3 3"/>
<line x1="596.2" y1="26" x2="596.2" y2="176" stroke="currentColor" stroke-opacity="0.12"/>
<text x="596.2" y="190" font-size="11" fill="currentColor" text-anchor="middle">K1</text>
<line x1="653.8" y1="26" x2="653.8" y2="176" stroke="currentColor" stroke-opacity="0.12"/>
<text x="653.8" y="190" font-size="11" fill="currentColor" text-anchor="middle">K2</text>
<polyline points="510.0,154.6 596.2,154.6 653.8,47.4 740.0,47.4" fill="none" stroke="#3d6ea5" stroke-width="2.6" stroke-linejoin="round"/>
<text x="625" y="16" font-size="12.5" font-weight="bold" fill="currentColor" text-anchor="middle">Bull call spread</text>
<text x="514" y="39" font-size="10.5" fill="currentColor" fill-opacity="0.75">long call K1, short call K2</text>
<rect x="760" y="26" width="230" height="150" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<line x1="760" y1="154.6" x2="990" y2="154.6" stroke="currentColor" stroke-opacity="0.35" stroke-dasharray="3 3"/>
<line x1="846.2" y1="26" x2="846.2" y2="176" stroke="currentColor" stroke-opacity="0.12"/>
<text x="846.2" y="190" font-size="11" fill="currentColor" text-anchor="middle">K1</text>
<line x1="903.8" y1="26" x2="903.8" y2="176" stroke="currentColor" stroke-opacity="0.12"/>
<text x="903.8" y="190" font-size="11" fill="currentColor" text-anchor="middle">K2</text>
<polyline points="760.0,47.4 846.2,47.4 903.8,154.6 990.0,154.6" fill="none" stroke="#3d6ea5" stroke-width="2.6" stroke-linejoin="round"/>
<text x="875" y="16" font-size="12.5" font-weight="bold" fill="currentColor" text-anchor="middle">Bear put spread</text>
<text x="764" y="39" font-size="10.5" fill="currentColor" fill-opacity="0.75">long put K2, short put K1</text>
<rect x="10" y="226" width="230" height="150" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<line x1="10" y1="356.0" x2="240" y2="356.0" stroke="currentColor" stroke-opacity="0.35" stroke-dasharray="3 3"/>
<line x1="96.2" y1="226" x2="96.2" y2="376" stroke="currentColor" stroke-opacity="0.12"/>
<text x="96.2" y="390" font-size="11" fill="currentColor" text-anchor="middle">K1</text>
<line x1="125.0" y1="226" x2="125.0" y2="376" stroke="currentColor" stroke-opacity="0.12"/>
<text x="125.0" y="390" font-size="11" fill="currentColor" text-anchor="middle">K2</text>
<line x1="153.8" y1="226" x2="153.8" y2="376" stroke="currentColor" stroke-opacity="0.12"/>
<text x="153.8" y="390" font-size="11" fill="currentColor" text-anchor="middle">K3</text>
<polyline points="10.0,356.0 96.2,356.0 125.0,256.0 153.8,356.0 240.0,356.0" fill="none" stroke="#3d6ea5" stroke-width="2.6" stroke-linejoin="round"/>
<text x="125" y="216" font-size="12.5" font-weight="bold" fill="currentColor" text-anchor="middle">Butterfly</text>
<text x="14" y="239" font-size="10.5" fill="currentColor" fill-opacity="0.75">+call K1, -2 call K2, +call K3</text>
<rect x="260" y="226" width="230" height="150" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<line x1="260" y1="363.5" x2="490" y2="363.5" stroke="currentColor" stroke-opacity="0.35" stroke-dasharray="3 3"/>
<line x1="375.0" y1="226" x2="375.0" y2="376" stroke="currentColor" stroke-opacity="0.12"/>
<text x="375.0" y="390" font-size="11" fill="currentColor" text-anchor="middle">K</text>
<polyline points="260.0,238.5 375.0,363.5 490.0,238.5" fill="none" stroke="#3d6ea5" stroke-width="2.6" stroke-linejoin="round"/>
<text x="375" y="216" font-size="12.5" font-weight="bold" fill="currentColor" text-anchor="middle">Straddle</text>
<text x="264" y="239" font-size="10.5" fill="currentColor" fill-opacity="0.75">long call K + long put K</text>
<rect x="510" y="226" width="230" height="150" fill="none" stroke="currentColor" stroke-opacity="0.4"/>
<line x1="510" y1="362.4" x2="740" y2="362.4" stroke="currentColor" stroke-opacity="0.35" stroke-dasharray="3 3"/>
<line x1="625.0" y1="226" x2="625.0" y2="376" stroke="currentColor" stroke-opacity="0.12"/>
<text x="625.0" y="390" font-size="11" fill="currentColor" text-anchor="middle">K</text>
<polyline points="510.0,362.4 625.0,362.4" fill="none" stroke="#3d6ea5" stroke-width="2.6" stroke-linejoin="round"/>
<polyline points="625.0,248.7 740.0,248.7" fill="none" stroke="#3d6ea5" stroke-width="2.6" stroke-linejoin="round"/>
<text x="625" y="216" font-size="12.5" font-weight="bold" fill="currentColor" text-anchor="middle">Binary (cash-or-nothing) call</text>
<text x="514" y="239" font-size="10.5" fill="currentColor" fill-opacity="0.75">pays Q if S_T > K</text>
<text x="766" y="246" font-size="12.5" font-weight="bold" fill="currentColor">Path / time dependent</text>
<text x="766" y="272" font-size="12" fill="currentColor" fill-opacity="0.85">Barrier, Asian and chooser</text>
<text x="766" y="292" font-size="12" fill="currentColor" fill-opacity="0.85">options have no single payoff</text>
<text x="766" y="312" font-size="12" fill="currentColor" fill-opacity="0.85">curve in S_T alone - they depend</text>
<text x="766" y="332" font-size="12" fill="currentColor" fill-opacity="0.85">on the whole price path (barrier,</text>
<text x="766" y="352" font-size="12" fill="currentColor" fill-opacity="0.85">Asian) or on a decision made</text>
<text x="766" y="372" font-size="12" fill="currentColor" fill-opacity="0.85">before maturity (chooser).</text>
<text x="500" y="426" font-size="12" fill="currentColor" text-anchor="middle">horizontal axis: stock price at maturity S_T;  vertical axis: value of the position at maturity (premiums excluded);  dashed grey: the stock alone</text></svg>
</p>

**Two identities worth remembering.**
- *In–out parity:* a knock-in plus the matching knock-out always pay exactly the vanilla payoff (exactly one of
  them is alive at maturity), so their prices add up to the vanilla price.
- *Chooser decomposition:* at time $t_1$ put–call parity gives $c-p=S_{t_1}-Ke^{-r(T-t_1)}$, so
  $\max(c,p)=c+\big(Ke^{-r(T-t_1)}-S_{t_1}\big)^+$. The chooser is therefore **a call maturing at $T$ plus a put
  maturing at $t_1$ with strike $Ke^{-r(T-t_1)}$**, and
$$\text{chooser}=c\big(S,K,T\big)+p\big(S,\,Ke^{-r(T-t_1)},\,t_1\big).$$

In [19]:
import math, random

# --- payoffs at maturity (premiums excluded) ---
pos = lambda x: max(x, 0.0)
K, K1, K2, Q = 100.0, 90.0, 110.0, 1.0
payoff = {
    "covered call":    lambda s: s - pos(s - K),
    "protective put":  lambda s: s + pos(K - s),
    "bull call spread": lambda s: pos(s - K1) - pos(s - K2),
    "bear put spread":  lambda s: pos(K2 - s) - pos(K1 - s),
    "butterfly":       lambda s: pos(s - K1) - 2 * pos(s - K) + pos(s - K2),
    "straddle":        lambda s: pos(s - K) + pos(K - s),
    "binary call (Q=1)": lambda s: Q if s > K else 0.0,
}
grid = (70, 90, 100, 110, 130)
print("payoff at maturity for S_T =", grid)
for name, f in payoff.items():
    print(f"  {name:<18}" + "".join(f"{f(s):9.2f}" for s in grid))

# --- what each costs today (Black-Scholes, S=100, r=5%, sigma=25%, one year) ---
S0, r, sigma, tau = 100.0, 0.05, 0.25, 1.0
c = lambda k: bs(S0, k, r, sigma, tau)
p = lambda k: bs(S0, k, r, sigma, tau, call=False)
d2 = d1_d2(S0, K, r, sigma, tau)[1]
cost = {"covered call (buy stock, sell call)": S0 - c(K), "protective put (buy stock, buy put)": S0 + p(K),
        "bull call spread": c(K1) - c(K2), "bear put spread": p(K2) - p(K1),
        "butterfly": c(K1) - 2 * c(K) + c(K2), "straddle": c(K) + p(K),
        "binary call (pays 1)": math.exp(-r * tau) * N(d2)}
print("\nprice today (S=100, r=5%, sigma=25%, tau=1):")
for k, v in cost.items():
    print(f"  {k:<38}{v:9.4f}")

# --- the exotics: price today for K=100 ---
def down_out_call(S, K, H, r, sigma, T):
    """Closed form (Merton / Reiner-Rubinstein, H <= K, no dividends): vanilla - knock-in."""
    lam = (r + sigma ** 2 / 2) / sigma ** 2
    y = math.log(H * H / (S * K)) / (sigma * math.sqrt(T)) + lam * sigma * math.sqrt(T)
    knock_in = S * (H / S) ** (2 * lam) * N(y) - K * math.exp(-r * T) * (H / S) ** (2 * lam - 2) * N(y - sigma * math.sqrt(T))
    return bs(S, K, r, sigma, T) - knock_in, knock_in

def simulate_exotics(S, K, H, r, sigma, T, paths=40000, steps=50, seed=9):
    """Monte Carlo of a down-and-out call (exact Brownian-bridge barrier test) and an arithmetic Asian call."""
    rng = random.Random(seed); dt = T / steps
    barrier = asian = 0.0
    for _ in range(paths):
        s = S; alive = 1.0; total = 0.0
        for _ in range(steps):
            s2 = s * math.exp((r - sigma ** 2 / 2) * dt + sigma * math.sqrt(dt) * rng.gauss(0, 1))
            if alive and s2 > H:
                alive *= 1 - math.exp(-2 * math.log(s / H) * math.log(s2 / H) / (sigma ** 2 * dt))
            else:
                alive = 0.0
            s = s2; total += s
        barrier += alive * pos(s - K); asian += pos(total / steps - K)
    disc = math.exp(-r * T) / paths
    return barrier * disc, asian * disc

H, t1 = 85.0, 0.25
do_cf, di_cf = down_out_call(S0, K, H, r, sigma, tau)
do_mc, asian_mc = simulate_exotics(S0, K, H, r, sigma, tau)
chooser = bs(S0, K, r, sigma, tau) + bs(S0, K * math.exp(-r * (tau - t1)), r, sigma, t1, call=False)
rng = random.Random(2); acc = 0.0
for _ in range(60000):                                   # check the chooser by pricing max(c,p) at t1 directly
    s1 = S0 * math.exp((r - sigma ** 2 / 2) * t1 + sigma * math.sqrt(t1) * rng.gauss(0, 1))
    acc += max(bs(s1, K, r, sigma, tau - t1), bs(s1, K, r, sigma, tau - t1, call=False))
chooser_mc = math.exp(-r * t1) * acc / 60000
print(f"\nexotic prices (K=100):")
print(f"  vanilla call                      {bs(S0, K, r, sigma, tau):8.4f}")
print(f"  down-and-out call, H=85           {do_cf:8.4f}  (Monte Carlo {do_mc:.4f});   knock-in {di_cf:.4f}; in + out = vanilla")
print(f"  arithmetic Asian call (50 dates)  {asian_mc:8.4f}  (averaging cuts the price roughly in half)")
print(f"  chooser (decide at t1 = 0.25)     {chooser:8.4f}  (Monte Carlo {chooser_mc:.4f});   straddle {c(K) + p(K):.4f}")

payoff at maturity for S_T = (70, 90, 100, 110, 130)
  covered call          70.00    90.00   100.00   100.00   100.00
  protective put       100.00   100.00   100.00   110.00   130.00
  bull call spread       0.00     0.00    10.00    20.00    20.00
  bear put spread       20.00    20.00    10.00     0.00     0.00
  butterfly              0.00     0.00    10.00     0.00     0.00
  straddle              30.00    10.00     0.00    10.00    30.00
  binary call (Q=1)      0.00     0.00     0.00     1.00     1.00

price today (S=100, r=5%, sigma=25%, tau=1):
  covered call (buy stock, sell call)     87.6640
  protective put (buy stock, buy put)    107.4589
  bull call spread                        10.1144
  bear put spread                          8.9102
  butterfly                                1.4951
  straddle                                19.7949
  binary call (pays 1)                     0.5040

exotic prices (K=100):
  vanilla call                       12.3360
  down-and-out call,

### 6.3.1 Bull spread

**Question.** What are the price boundaries of a bull call spread? Tabulate its cash flows at time $0$ and at maturity.

**Construction.** Choose strikes $K_1<K_2$ and **buy** a call struck at $K_1$ (cost $c_1$) and **sell** a call struck
at $K_2$ (receive $c_2$), same maturity $T$. The spread costs $c_1-c_2$ today.

**Cash-flow table.** Add the two legs state by state:

| position | time $0$ | $S_T\le K_1$ | $K_1<S_T<K_2$ | $S_T\ge K_2$ |
|---|:---:|:---:|:---:|:---:|
| long call $K_1$ | $-c_1$ | $0$ | $S_T-K_1$ | $S_T-K_1$ |
| short call $K_2$ | $+c_2$ | $0$ | $0$ | $-(S_T-K_2)$ |
| **bull spread** | $-(c_1-c_2)$ | $0$ | $S_T-K_1$ | $K_2-K_1$ |

The payoff is a ramp: nothing below $K_1$, rising one-for-one in between, and capped at $K_2-K_1$ above $K_2$. It is
**never negative, and never more than $K_2-K_1$**.

**Price bounds.**
$$\boxed{\ 0\;<\;c_1-c_2\;<\;(K_2-K_1)\,e^{-r\tau}\ }.$$

**Step 1 — lower bound (a payoff that cannot lose is not free).** Since the spread pays $\ge0$ in every state, its
price today must be $\ge0$; and since it pays strictly more than $0$ whenever $S_T>K_1$ (an event of positive
probability), the price is **strictly positive**. *If instead $c_1-c_2\le0$* the spread would be free (or pay you to
hold it) with a payoff that is never negative and sometimes positive — an arbitrage. Equivalently $c_1>c_2$: a call
with a lower strike is worth more, because $(S_T-K_1)^+\ge(S_T-K_2)^+$ in every state.

**Step 2 — upper bound (it cannot be worth more than its best payoff).** The spread pays at most $K_2-K_1$, so it
cannot cost more than the present value of that amount, $(K_2-K_1)e^{-r\tau}$. It pays *strictly less* whenever
$S_T<K_2$ (positive probability), hence the inequality is **strict**. *If instead $c_1-c_2\ge(K_2-K_1)e^{-r\tau}$*,
sell the spread and lend the proceeds: at maturity you owe at most $K_2-K_1$ but hold
$(c_1-c_2)e^{r\tau}\ge K_2-K_1$ — a profit that is never negative and strictly positive when $S_T<K_2$.

**Step 3 — a useful reading.** Dividing by the strike gap, $\dfrac{c_1-c_2}{K_2-K_1}\approx-\dfrac{\partial c}{\partial K}
=e^{-r\tau}N(d_2)$, the discounted risk-neutral probability of finishing in the money: a narrow bull spread is a
**digital option in disguise** (made exact in 6.3.3). The bear put spread has the mirror-image bounds
$0<p_2-p_1<(K_2-K_1)e^{-r\tau}$.

In [20]:
import math

def bull_spread_cost(S, K1, K2, r, sigma, tau):
    return bs(S, K1, r, sigma, tau) - bs(S, K2, r, sigma, tau)

S0, r, sigma, tau, K1, K2 = 100.0, 0.05, 0.25, 1.0, 95.0, 110.0
c1, c2 = bs(S0, K1, r, sigma, tau), bs(S0, K2, r, sigma, tau)
upper = (K2 - K1) * math.exp(-r * tau)
print(f"c1 = c(K1={K1:.0f}) = {c1:.4f},  c2 = c(K2={K2:.0f}) = {c2:.4f}")
print(f"bull spread cost c1 - c2 = {c1 - c2:.4f}   bounds: 0 < cost < (K2-K1)e^(-r tau) = {upper:.4f}")

print("\ncash flows (time 0 = -(c1-c2); maturity by state):")
print(f"{'S_T':>6} | {'long call K1':>12} {'short call K2':>14} | {'spread payoff':>13} | {'net P&L':>8}")
for ST in (80, 95, 100, 105, 110, 125):
    a, b = max(ST - K1, 0), -max(ST - K2, 0)
    print(f"{ST:>6} | {a:12.2f} {b:14.2f} | {a + b:13.2f} | {a + b - (c1 - c2) * math.exp(r * tau):8.2f}")

# the bounds hold for every strike pair; and quotes outside the bounds are arbitrages
pairs = [(a, b) for a in range(60, 140, 5) for b in range(a + 5, 160, 5)]
ok = all(0 < bull_spread_cost(S0, a, b, r, sigma, tau) < (b - a) * math.exp(-r * tau) for a, b in pairs)
print(f"\nbounds hold for all {len(pairs)} strike pairs checked:", ok)
def classify(c1, c2, K1, K2, r, tau):
    up = (K2 - K1) * math.exp(-r * tau)
    if c1 - c2 <= 0:   return "ARBITRAGE: buy the spread - it costs nothing/pays you, and never loses"
    if c1 - c2 >= up:  return "ARBITRAGE: sell the spread and lend - the proceeds exceed its largest payoff"
    return "no arbitrage"
for q in [(5.0, 6.0), (14.0, 2.0), (9.0, 4.0)]:
    print(f"  quotes c(100)={q[0]}, c(110)={q[1]}, r=5%, tau=1:  {classify(q[0], q[1], 100, 110, 0.05, 1.0)}")

c1 = c(K1=95) = 15.0471,  c2 = c(K2=110) = 8.0264
bull spread cost c1 - c2 = 7.0207   bounds: 0 < cost < (K2-K1)e^(-r tau) = 14.2684

cash flows (time 0 = -(c1-c2); maturity by state):
   S_T | long call K1  short call K2 | spread payoff |  net P&L
    80 |         0.00           0.00 |          0.00 |    -7.38
    95 |         0.00           0.00 |          0.00 |    -7.38
   100 |         5.00           0.00 |          5.00 |    -2.38
   105 |        10.00           0.00 |         10.00 |     2.62
   110 |        15.00          -0.00 |         15.00 |     7.62
   125 |        30.00         -15.00 |         15.00 |     7.62

bounds hold for all 184 strike pairs checked: True
  quotes c(100)=5.0, c(110)=6.0, r=5%, tau=1:  ARBITRAGE: buy the spread - it costs nothing/pays you, and never loses
  quotes c(100)=14.0, c(110)=2.0, r=5%, tau=1:  ARBITRAGE: sell the spread and lend - the proceeds exceed its largest payoff
  quotes c(100)=9.0, c(110)=4.0, r=5%, tau=1:  no arbitrage


### 6.3.2 Straddle

**Question.** What is a straddle, and when would you buy one?

**Definition.** A **long straddle** is a call and a put with the **same strike $K$ and maturity**:
$$\text{payoff}=(S_T-K)^+ +(K-S_T)^+=\lvert S_T-K\rvert .$$
It pays the *size* of the move away from the strike and ignores its direction. With total premium
$\Pi=c+p$ today, the profit is $\lvert S_T-K\rvert-\Pi e^{r\tau}$, so you make money only if the stock finishes
**outside the break-evens** $K\pm\Pi e^{r\tau}$ (ignoring interest, $K\pm\Pi$); the most you can lose is the premium
(if $S_T=K$), the most you can win is unlimited upward and $K-\Pi$ downward.

**When to buy it.** Buy a straddle when you expect a **large move but do not know the direction** — before an
earnings release, a court ruling, a central-bank decision, an election — and, more precisely, when you think the
**realised volatility will exceed the implied volatility** you are paying for. At the money the straddle is
priced at roughly
$$\Pi\approx2\times0.4\,S\sigma\sqrt\tau=0.8\,S\,\sigma\sqrt\tau$$
(twice the ATM call estimate of 6.2), so it is a direct purchase of "$\sigma\sqrt\tau$": the market's expected move.
Its Greeks tell the same story: delta $\approx0$ (no directional view), **positive gamma and vega, negative
theta** — you pay time decay every day for the right to profit from large swings. Delta-hedged, the straddle's
profit over a short interval is
$$d\Pi_{\text{hedged}}\approx\tfrac12\,\Gamma\,S^{2}\big(\sigma_{\text{realised}}^{2}-\sigma_{\text{implied}}^{2}\big)\,dt,$$
a bet on realised against implied variance.

#### Comparison with volatility and variance swaps

A straddle is a *crude* volatility bet. The cleaner instruments are the swaps:

- **Variance swap:** pays $N_{\text{var}}\big(\sigma_R^{2}-K_{\text{var}}\big)$ at maturity, where $\sigma_R^2$ is the
  annualised *realised* variance of daily log-returns over the life of the swap and $K_{\text{var}}$ is the strike
  fixed so the swap costs nothing today.
- **Volatility swap:** pays $N_{\text{vol}}\big(\sigma_R-K_{\text{vol}}\big)$ — linear in volatility instead of
  variance.

**Why the straddle is the poorer volatility bet.** Its exposure depends on where the stock *is*. The straddle's
**dollar gamma** $S^{2}\Gamma$ is huge at the strike and fades away from it; once the stock has moved far from $K$
the position stops responding to volatility, and the straddle also *only sees the terminal price*, not the path.

**How a variance swap fixes it — replication.** By Itô on $\ln S$, $d\ln S=(r-\tfrac12\sigma^2)dt+\sigma\,dW$, so
$$\int_0^{\tau}\sigma^{2}dt=2\int_0^{\tau}\frac{dS}{S}-2\ln\frac{S_T}{S_0}\quad(\text{up to the drift }r),$$
i.e. realised variance equals a **dynamic stock position** (the $dS/S$ term, hedged with futures) minus twice a
**log contract**. For any smooth payoff $f$ (Carr–Madan), with forward $F$,
$$f(S_T)=f(F)+f'(F)(S_T-F)+\int_0^{F}f''(K)(K-S_T)^{+}dK+\int_{F}^{\infty}f''(K)(S_T-K)^{+}dK .$$
Taking $f=-\ln$ gives $f''(K)=1/K^{2}$, so the log contract is a **strip of out-of-the-money options with weights
$1/K^{2}$**, and the fair strike is
$$K_{\text{var}}=\frac{2}{\tau}\,e^{r\tau}\Big[\int_0^{F}\frac{p(K)}{K^{2}}dK+\int_{F}^{\infty}\frac{c(K)}{K^{2}}dK\Big].$$
Each option in the strip has gamma $\Gamma_K(S)$ and, because the weights are $1/K^2$, the **dollar gamma of the whole
strip is the same at every stock price**, $S^2\int\Gamma_K(S)\,K^{-2}dK=e^{-r\tau}$ — a flat exposure to variance,
whereas the straddle's dollar gamma is a spike. Under Black–Scholes the formula returns exactly $K_{\text{var}}=\sigma^{2}$.

| | straddle | variance swap | volatility swap |
|---|---|---|---|
| payoff | $\lvert S_T-K\rvert$ | $N_{\text{var}}(\sigma_R^{2}-K_{\text{var}})$ | $N_{\text{vol}}(\sigma_R-K_{\text{vol}})$ |
| what it depends on | terminal price only | realised variance along the **whole path** | realised volatility along the whole path |
| exposure to volatility | peaks at the strike, fades as $S$ moves away | **constant** whatever the stock does (flat dollar gamma) | constant in volatility, not in variance |
| upfront cost | premium $\approx0.8S\sigma\sqrt\tau$ | **zero** (strike set so value is $0$) | zero |
| replication | two options | strip of options with weights $1/K^{2}$ plus dynamic stock | no static replication |
| strike vs implied vol | — | $K_{\text{var}}=\sigma_{\text{imp}}^{2}$ | $K_{\text{vol}}<\sqrt{K_{\text{var}}}$ (Jensen: $E[\sigma_R]\le\sqrt{E[\sigma_R^2]}$) |
| main risk | decay and a flat stock | large realised variance spikes (it is convex in volatility) | smaller convexity |

In [21]:
import math, random

S0, K, r, sigma, tau = 100.0, 100.0, 0.05, 0.25, 1.0
cc, pp = bs(S0, K, r, sigma, tau), bs(S0, K, r, sigma, tau, call=False)
prem = cc + pp
print(f"ATM straddle premium = {prem:.4f}   (estimate 0.8*S*sigma*sqrt(tau) = {0.8 * S0 * sigma * math.sqrt(tau):.4f})")
print(f"break-evens at maturity: {K - prem * math.exp(r * tau):.2f} and {K + prem * math.exp(r * tau):.2f}")
print("profit at maturity  S_T:      " + "".join(f"{s:8d}" for s in (60, 80, 90, 100, 110, 120, 140)))
print("                    profit: " + "".join(f"{abs(s - K) - prem * math.exp(r * tau):8.2f}" for s in (60, 80, 90, 100, 110, 120, 140)))

# ---- variance swap: replication gives K_var = sigma^2 ----
def fair_variance_strike(S, r, sigma, T, lo=1e-2, hi=3000.0, n=20000):
    F = S * math.exp(r * T); a, b = math.log(lo), math.log(hi); h = (b - a) / n; tot = 0.0
    for i in range(n):
        k = math.exp(a + (i + 0.5) * h)                              # integrate on a log grid: dK = K dlnK
        opt = bs(S, k, r, sigma, T, call=False) if k < F else bs(S, k, r, sigma, T, call=True)
        tot += opt / (k * k) * k * h
    return 2 / T * math.exp(r * T) * tot

print(f"\nvariance-swap strike by the 1/K^2 option strip: {fair_variance_strike(S0, r, sigma, tau):.6f}   sigma^2 = {sigma ** 2:.6f}")

# ---- dollar gamma: straddle (spike) vs the 1/K^2 strip (flat) ----
def strip_dollar_gamma(S, r, sigma, T, n=4000):
    a, b = math.log(1e-2), math.log(5000.0); h = (b - a) / n; tot = 0.0
    for i in range(n):
        k = math.exp(a + (i + 0.5) * h)
        gam = norm_pdf(d1_d2(S, k, r, sigma, T)[0]) / (S * sigma * math.sqrt(T))
        tot += gam * S * S / (k * k) * k * h
    return tot
straddle_dg = lambda S: 2 * norm_pdf(d1_d2(S, K, r, sigma, tau)[0]) / (S * sigma * math.sqrt(tau)) * S * S
print("\ndollar gamma S^2*Gamma at spot 60 / 100 / 150:")
print("  straddle      :", "  ".join(f"{straddle_dg(s):8.3f}" for s in (60, 100, 150)), "   <- a spike at the strike")
print("  1/K^2 strip   :", "  ".join(f"{strip_dollar_gamma(s, r, sigma, tau):8.3f}" for s in (60, 100, 150)),
      f"   <- flat, equal to e^(-r tau) = {math.exp(-r * tau):.3f}")

# ---- realised variance vs realised volatility (weekly observations, true vol = 25%) ----
rng = random.Random(4); steps, paths = 52, 20000; dt = tau / steps
sv = sd = 0.0
for _ in range(paths):
    rv = sum((((r - sigma ** 2 / 2) * dt + sigma * math.sqrt(dt) * rng.gauss(0, 1))) ** 2 for _ in range(steps)) / tau
    sv += rv; sd += math.sqrt(rv)
print(f"\nrealised variance averages {sv / paths:.5f} (= sigma^2 = {sigma ** 2:.4f}); realised volatility averages "
      f"{sd / paths:.5f} < sigma = {sigma}  ->  K_vol < sqrt(K_var)")

ATM straddle premium = 19.7949   (estimate 0.8*S*sigma*sqrt(tau) = 20.0000)
break-evens at maturity: 79.19 and 120.81
profit at maturity  S_T:            60      80      90     100     110     120     140
                    profit:    19.19   -0.81  -10.81  -20.81  -10.81   -0.81   19.19

variance-swap strike by the 1/K^2 option strip: 0.062500   sigma^2 = 0.062500

dollar gamma S^2*Gamma at spot 60 / 100 / 150:
  straddle      :   43.753   302.736    71.952    <- a spike at the strike
  1/K^2 strip   :    0.951     0.951     0.951    <- flat, equal to e^(-r tau) = 0.951

realised variance averages 0.06249 (= sigma^2 = 0.0625); realised volatility averages 0.24882 < sigma = 0.25  ->  K_vol < sqrt(K_var)


### 6.3.3 Binary (cash-or-nothing) options

**Question.** A cash-or-nothing European call pays a fixed cash amount $Q$ at $T$ if $S_T>K$ and $0$ otherwise.
What is its price if the stock follows geometric Brownian motion? How do we hedge it, and what limits the hedge?

#### Pricing, step by step

**Step 1 — the pricing principle.** The price is the discounted risk-neutral expectation of the payoff:
$$V=e^{-r\tau}\,E^{Q}\big[\,Q\cdot\mathbf 1_{S_T>K}\big]=Q\,e^{-r\tau}\;P^{Q}(S_T>K).$$
The whole problem is the *risk-neutral probability of finishing above $K$*.

**Step 2 — the risk-neutral stock.** Under $Q$, $dS=rS\,dt+\sigma S\,dW$. Itô on $\ln S$ gives
$d\ln S=\big(r-\tfrac12\sigma^2\big)dt+\sigma\,dW$, hence
$$S_T=S\,\exp\!\Big[\big(r-\tfrac12\sigma^{2}\big)\tau+\sigma\sqrt\tau\,Z\Big],\qquad Z\sim N(0,1).$$

**Step 3 — rewrite the event in terms of $Z$.** Taking logarithms,
$$S_T>K\iff\big(r-\tfrac12\sigma^{2}\big)\tau+\sigma\sqrt\tau\,Z>\ln\frac KS\iff Z>-\,\frac{\ln(S/K)+\big(r-\tfrac12\sigma^{2}\big)\tau}{\sigma\sqrt\tau}=-d_2 .$$

**Step 4 — the probability.** By the symmetry of the normal law, $P(Z>-d_2)=P(Z<d_2)=N(d_2)$.

**Step 5 — assemble.**
$$\boxed{\,V_{\text{call}}=Q\,e^{-r\tau}N(d_2)\,},\qquad V_{\text{put}}=Q\,e^{-r\tau}N(-d_2),\qquad
d_2=\frac{\ln(S/K)+(r-\tfrac12\sigma^{2})\tau}{\sigma\sqrt\tau}.$$
Check: the digital call and put together pay $Q$ for certain, and $Qe^{-r\tau}\big[N(d_2)+N(-d_2)\big]=Qe^{-r\tau}$ is exactly
the price of a zero-coupon bond. Note also that the Black–Scholes call splits as
$c=\underbrace{S\,N(d_1)}_{\text{asset-or-nothing}}-K\underbrace{e^{-r\tau}N(d_2)}_{\text{cash-or-nothing paying }1}$ — the
digital is one of the two atoms of the vanilla call.

**Step 6 — the same price from a derivative in the strike.** A call's payoff has slope $-1$ in $K$ on
$\{S_T>K\}$, so $-\partial c/\partial K=e^{-r\tau}P^{Q}(S_T>K)$ (Breeden–Litzenberger, 6.2). Differentiate $c=SN(d_1)-Ke^{-r\tau}N(d_2)$
with $\partial d_1/\partial K=\partial d_2/\partial K=-\dfrac1{K\sigma\sqrt\tau}$:
$$\frac{\partial c}{\partial K}=\underbrace{S\,N'(d_1)\Big(\!-\tfrac{1}{K\sigma\sqrt\tau}\Big)+Ke^{-r\tau}N'(d_2)\tfrac{1}{K\sigma\sqrt\tau}}_{=0\ \text{since }SN'(d_1)=Ke^{-r\tau}N'(d_2)}-e^{-r\tau}N(d_2)=-e^{-r\tau}N(d_2).$$
Hence $V_{\text{call}}=-Q\,\dfrac{\partial c}{\partial K}$: **a digital is a limiting bull spread** (this is the bridge to hedge 2).

#### The Greeks of a digital

**Delta.** Differentiating $N(d_2)$ with the chain rule and $\partial d_2/\partial S=\dfrac1{S\sigma\sqrt\tau}$,
$$\Delta=\frac{\partial V}{\partial S}=Q\,e^{-r\tau}\,\frac{N'(d_2)}{S\,\sigma\sqrt\tau}>0 .$$
**Gamma.** Differentiating again, using $N''(x)=-xN'(x)$ and $d_2+\sigma\sqrt\tau=d_1$,
$$\Gamma=-\,Q\,e^{-r\tau}\,\frac{N'(d_2)\,d_1}{S^{2}\sigma^{2}\tau},$$
which **changes sign** at $d_1=0$, i.e. at the stock price $Ke^{-(r+\sigma^2/2)\tau}$ just *below* the strike: gamma is positive
further below and negative near and above the strike — unlike a vanilla option, whose gamma is never negative.
At the money $N'(d_2)\to N'(0)\approx0.4$, so $\Delta\approx\dfrac{0.4\,Q}{S\sigma\sqrt\tau}\to\infty$ as $\tau\to0$.

#### Hedge 1 — standard delta hedging

A trader who **sells** the digital hedges by **holding $\Delta$ shares** (buying as the stock rises toward $K$, selling as it
falls), financed with cash, and rebalancing as $S$ and $\tau$ change. This replicates the payoff *in the limit of
continuous rebalancing*. **Limitations:**
1. **Delta explodes near the strike at expiry** ($\Delta\propto1/\sqrt\tau$): the output below shows the shares needed
   per unit of payoff growing about twenty-fold between one year and one day to expiry — with one day left a
   claim paying 1,000,000 dollars needs roughly 300,000 shares, about 30 million dollars of stock, to hedge.
2. The payoff is **discontinuous**: a tiny move across $K$ swings the claim by the full $Q$ (**pin risk**), so the hedge
   cannot track it, and gamma flips sign.
3. **Hedge error shrinks slowly** with the number of rebalances (roughly $n^{-1/4}$ in the simulation below, compared with
   $n^{-1/2}$ for a vanilla call), and each rebalance costs money (bid-ask, impact).
4. It is only as good as the model: it assumes constant $\sigma$ and a continuous path.

#### Hedge 2 — a bull call spread of two calls

Replace the jump by a steep but continuous ramp. **Buy $Q/\varepsilon$ calls struck at $K-\varepsilon$ and sell $Q/\varepsilon$
calls struck at $K$.** Its payoff in each state:

| state | $S_T\le K-\varepsilon$ | $K-\varepsilon<S_T<K$ | $S_T\ge K$ |
|---|:---:|:---:|:---:|
| bull spread of $Q/\varepsilon$ calls | $0$ | $\dfrac{Q}{\varepsilon}(S_T-K+\varepsilon)\in(0,Q)$ | $Q$ |
| cash-or-nothing call | $0$ | $0$ | $Q$ |

The spread pays **at least** the digital in every state, so it is a **super-replication**: a *static* hedge that needs no
rebalancing. It costs
$$\frac{Q}{\varepsilon}\big[c(K-\varepsilon)-c(K)\big]\ \xrightarrow[\varepsilon\to0]{}\ -Q\,\frac{\partial c}{\partial K}=Q\,e^{-r\tau}N(d_2),$$
converging to the digital price **from above** (call prices are convex in $K$). **Limitations:** (i) the hedge pays more
than the digital when $S_T\in(K-\varepsilon,K)$, so the seller pays an *over-hedge premium* of order $\varepsilon$; (ii)
shrinking $\varepsilon$ to cut that cost needs a position $Q/\varepsilon$ in each call — huge, with bid-ask costs and
limited by the strikes actually listed; (iii) it still needs the digital's price to be the *model* price, but a useful
side-benefit is that the spread automatically embeds the **volatility smile** (the digital's price is $-\partial c/\partial K$
of the *market* call curve, including the skew term).

In [22]:
import math, random

def digital_call(Q, S, K, r, sigma, tau):
    d2 = d1_d2(S, K, r, sigma, tau)[1]
    return Q * math.exp(-r * tau) * N(d2)

def digital_delta(Q, S, K, r, sigma, tau):
    d2 = d1_d2(S, K, r, sigma, tau)[1]
    return Q * math.exp(-r * tau) * norm_pdf(d2) / (S * sigma * math.sqrt(tau))

def digital_gamma(Q, S, K, r, sigma, tau):
    d1, d2 = d1_d2(S, K, r, sigma, tau)
    return -Q * math.exp(-r * tau) * norm_pdf(d2) * d1 / (S * S * sigma ** 2 * tau)

def bull_spread_hedge_cost(Q, S, K, eps, r, sigma, tau):
    """Cost of Q/eps calls at K-eps minus Q/eps calls at K (super-replicates the digital)."""
    return Q / eps * (bs(S, K - eps, r, sigma, tau) - bs(S, K, r, sigma, tau))

Q, S0, K, r, sigma, tau = 1.0, 100.0, 100.0, 0.05, 0.25, 1.0
price = digital_call(Q, S0, K, r, sigma, tau)
rng = random.Random(0); n = 300000
mc = math.exp(-r * tau) * sum(S0 * math.exp((r - sigma ** 2 / 2) * tau + sigma * math.sqrt(tau) * rng.gauss(0, 1)) > K for _ in range(n)) / n
print(f"digital call price = Q e^(-r tau) N(d2) = {price:.6f}   (Monte Carlo {mc:.6f})")
print(f"digital put + call = {price + Q * math.exp(-r * tau) * N(-d1_d2(S0, K, r, sigma, tau)[1]):.6f}  = zero-coupon bond Q e^(-r tau) = {Q * math.exp(-r * tau):.6f}")
print(f"-dc/dK (finite difference) = {-(bs(S0, K + 1e-4, r, sigma, tau) - bs(S0, K - 1e-4, r, sigma, tau)) / 2e-4:.6f}")

h = 1e-3
fd_delta = (digital_call(Q, S0 + h, K, r, sigma, tau) - digital_call(Q, S0 - h, K, r, sigma, tau)) / (2 * h)
fd_gamma = (digital_call(Q, S0 + 0.05, K, r, sigma, tau) - 2 * price + digital_call(Q, S0 - 0.05, K, r, sigma, tau)) / 0.05 ** 2
print(f"\ndelta  formula {digital_delta(Q, S0, K, r, sigma, tau):.6f}   finite difference {fd_delta:.6f}")
print(f"gamma  formula {digital_gamma(Q, S0, K, r, sigma, tau):+.6f}   finite difference {fd_gamma:+.6f}   (positive well below the strike, negative near and above it: "
      f"S=80 {digital_gamma(Q, 80, K, r, sigma, tau):+.5f}, S=90 {digital_gamma(Q, 90, K, r, sigma, tau):+.5f}, S=110 {digital_gamma(Q, 110, K, r, sigma, tau):+.5f}; "
      f"zero at d1 = 0, S = {K * math.exp(-(r + sigma ** 2 / 2) * tau):.1f})")

print("\nHedge 1 limitation - ATM digital delta (shares per $1 of payoff) as expiry nears:")
for lab, T in [("1 year", 1.0), ("3 months", 0.25), ("1 month", 1 / 12), ("1 week", 1 / 52), ("1 day", 1 / 365)]:
    print(f"  {lab:>8}: {digital_delta(1.0, 100, 100, r, sigma, T):.4f} shares   ->  ${1e6 * digital_delta(1.0, 100, 100, r, sigma, T) * 100:,.0f} of stock per $1,000,000 payout")

def delta_hedge_error(n, paths=800, mu=0.10, seed=3):
    """Sell the digital, hedge with delta shares rebalanced n times; returns (mean, std) of the final hedging P&L."""
    rng = random.Random(seed); dt = tau / n; out = []
    for _ in range(paths):
        s = S0; dl = digital_delta(Q, s, K, r, sigma, tau)
        cash = digital_call(Q, S0, K, r, sigma, tau) - dl * s     # premium received for the digital, minus the shares bought
        for i in range(1, n + 1):
            s *= math.exp((mu - sigma ** 2 / 2) * dt + sigma * math.sqrt(dt) * rng.gauss(0, 1)); cash *= math.exp(r * dt)
            if i < n:
                nd = digital_delta(Q, s, K, r, sigma, tau - i * dt); cash -= (nd - dl) * s; dl = nd
        out.append(cash + dl * s - (Q if s > K else 0.0))          # premium account + shares - payout owed
    m = sum(out) / paths
    return m, (sum((x - m) ** 2 for x in out) / paths) ** 0.5

print("\nHedge 1 limitation - delta-hedging error of a sold digital (ideal P&L = 0; digital price 0.504):")
prev = None
for n_ in (4, 16, 64, 252):
    m, sd = delta_hedge_error(n_)
    print(f"  {n_:>3} rebalances: mean {m:+.4f}, std {sd:.4f}" + (f"   (std shrank by x{prev / sd:.2f} for 4x the rebalances; vanilla: x2)" if prev else ""))
    prev = sd

print("\nHedge 2 - bull spread of Q/eps calls: cost converges to the digital price from above:")
for eps in (10, 5, 2, 1, 0.5, 0.1, 0.01):
    cst = bull_spread_hedge_cost(Q, S0, K, eps, r, sigma, tau)
    print(f"  eps = {eps:>5}:  cost {cst:.6f}   over-hedge premium {cst - price:+.6f}   contracts needed per $1 of payoff: {Q / eps:g}")

digital call price = Q e^(-r tau) N(d2) = 0.504049   (Monte Carlo 0.504963)
digital put + call = 0.951229  = zero-coupon bond Q e^(-r tau) = 0.951229
-dc/dK (finite difference) = 0.504049

delta  formula 0.015137   finite difference 0.015137
gamma  formula -0.000197   finite difference -0.000197   (positive well below the strike, negative near and above it: S=80 +0.00039, S=90 +0.00007, S=110 -0.00032; zero at d1 = 0, S = 92.2)

Hedge 1 limitation - ATM digital delta (shares per $1 of payoff) as expiry nears:
    1 year: 0.0151 shares   ->  $1,513,679 of stock per $1,000,000 payout
  3 months: 0.0315 shares   ->  $3,149,677 of stock per $1,000,000 payout
   1 month: 0.0550 shares   ->  $5,503,631 of stock per $1,000,000 payout
    1 week: 0.1150 shares   ->  $11,495,574 of stock per $1,000,000 payout
     1 day: 0.3048 shares   ->  $30,482,715 of stock per $1,000,000 payout

Hedge 1 limitation - delta-hedging error of a sold digital (ideal P&L = 0; digital price 0.504):
    4 rebalance

### 6.3.4 Exchange options

**Question.** Price an exchange option paying $\max(S_{T}^{1}-S_{T}^{2},\,0)$ at maturity $T$, where $S^1$ and $S^2$ are
non-dividend stocks following geometric Brownian motions with correlation $\rho$.

**Set-up.** Under the risk-neutral measure $Q$,
$$dS^{i}=rS^{i}dt+\sigma_i S^{i}\,dW^{i},\qquad dW^{1}dW^{2}=\rho\,dt,\qquad i=1,2 .$$
Define the **volatility of the ratio** $S^1/S^2$
$$\sigma^{2}\;=\;\sigma_1^{2}+\sigma_2^{2}-2\rho\,\sigma_1\sigma_2 .$$
The price is $V=e^{-r\tau}E^{Q}\big[(S^1_T-S^2_T)^{+}\big]$.

#### Solution 1 — Margrabe's formula

$$\boxed{\ V=S^{1}N(d_1)-S^{2}N(d_2),\qquad d_1=\frac{\ln\!\big(S^{1}/S^{2}\big)+\tfrac12\sigma^{2}\tau}{\sigma\sqrt\tau},\quad d_2=d_1-\sigma\sqrt\tau\ }$$
**Remarkably the interest rate $r$ does not appear**: both stocks grow at $r$, so the *difference* in growth is nil.

*Derivation by splitting the payoff.* Let $A=\{S^1_T>S^2_T\}$. Then
$$V=\underbrace{e^{-r\tau}E^{Q}\big[S^{1}_T\mathbf 1_A\big]}_{\text{term 1}}-\underbrace{e^{-r\tau}E^{Q}\big[S^{2}_T\mathbf 1_A\big]}_{\text{term 2}} .$$
Because $e^{-rt}S^{i}_t$ is a $Q$-martingale, define a new measure $Q^{i}$ by $\dfrac{dQ^{i}}{dQ}=\dfrac{e^{-r\tau}S^{i}_T}{S^{i}}$.
Then $e^{-r\tau}E^{Q}[S^i_T\mathbf 1_A]=S^{i}\,Q^{i}(A)$, so $V=S^{1}Q^{1}(A)-S^{2}Q^{2}(A)$, and it remains to find the
probability of $A$ under each measure. Write $Y_t=S^{1}_t/S^{2}_t$ so $A=\{Y_T>1\}$. By Itô on the logarithms,
$$d\ln Y=\tfrac12\big(\sigma_2^{2}-\sigma_1^{2}\big)dt+\sigma_1dW^{1}-\sigma_2dW^{2},$$
with $\operatorname{Var}(\sigma_1dW^{1}-\sigma_2dW^{2})=\sigma^{2}dt$. Girsanov under $Q^{1}$ shifts $dW^{1}\to dW^{1}+\sigma_1dt$ and
$dW^{2}\to dW^{2}+\rho\sigma_1dt$, adding $\sigma_1^{2}-\rho\sigma_1\sigma_2$ to the drift, which becomes
$\tfrac12\sigma_2^2-\tfrac12\sigma_1^2+\sigma_1^2-\rho\sigma_1\sigma_2=+\tfrac12\sigma^{2}$. Under $Q^{2}$ the shifts are $\rho\sigma_2$ and $\sigma_2$, adding
$\rho\sigma_1\sigma_2-\sigma_2^{2}$, giving drift $-\tfrac12\sigma^{2}$. So $\ln Y_T$ is normal with variance $\sigma^2\tau$ and mean
$\ln Y_0\pm\tfrac12\sigma^2\tau$, hence
$$Q^{1}(Y_T>1)=N\!\Big(\tfrac{\ln Y_0+\frac12\sigma^2\tau}{\sigma\sqrt\tau}\Big)=N(d_1),\qquad
Q^{2}(Y_T>1)=N\!\Big(\tfrac{\ln Y_0-\frac12\sigma^2\tau}{\sigma\sqrt\tau}\Big)=N(d_2),$$
and $V=S^1N(d_1)-S^2N(d_2)$.

#### Solution 2 — Itô's lemma and the martingale equations

**Step 1 — discounted prices are martingales.** Itô on $e^{-rt}S^{i}_t$: $d\big(e^{-rt}S^i\big)=e^{-rt}S^i\sigma_i\,dW^i$, which has **no drift**, so
each is a $Q$-martingale (the martingale criterion of 5.4). A derivative's discounted price is also a martingale,
so $V=e^{-r\tau}E^{Q}\big[(S^1_T-S^2_T)^+\big]$.

**Step 2 — turn the problem into one about a ratio.** Because the payoff is homogeneous of degree one,
$(S^1_T-S^2_T)^+=S^2_T\,(Y_T-1)^+$ with $Y=S^1/S^2$. Apply Itô to $Y=S^1/S^2$ (using $d(1/S^2)$ from 6.1.5-E):
$$\frac{dY}{Y}=\big(\sigma_2^{2}-\rho\sigma_1\sigma_2\big)dt+\sigma_1dW^{1}-\sigma_2dW^{2}.$$
The drift $\sigma_2^2-\rho\sigma_1\sigma_2$ is **not zero** (and $r$ cancelled), so $Y$ is not a $Q$-martingale.

**Step 3 — find the measure in which $Y$ *is* a martingale.** Use the **stock $S^2$ as the numeraire**: define $Q^{2}$ with
density process $\dfrac{e^{-rt}S^{2}_t}{S^2_0}$, a positive $Q$-martingale from Step 1. By Girsanov, under $Q^2$
$$dW^{1}=dW^{1,*}+\rho\sigma_2dt,\qquad dW^{2}=dW^{2,*}+\sigma_2dt,$$
and substituting into Step 2,
$$\frac{dY}{Y}=\big(\sigma_2^{2}-\rho\sigma_1\sigma_2\big)dt+\sigma_1\big(dW^{1,*}+\rho\sigma_2dt\big)-\sigma_2\big(dW^{2,*}+\sigma_2dt\big)
=\sigma_1dW^{1,*}-\sigma_2dW^{2,*},$$
since $\sigma_2^2-\rho\sigma_1\sigma_2+\rho\sigma_1\sigma_2-\sigma_2^2=0$. **$Y$ is a driftless geometric Brownian motion under $Q^2$ with volatility $\sigma$.**

**Step 4 — change measure in the price.** From the density, $E^{Q}\big[e^{-r\tau}S^2_TX\big]=S^{2}E^{Q^{2}}[X]$, so
$$V=E^{Q}\big[e^{-r\tau}S^2_T\,(Y_T-1)^+\big]=S^{2}\;E^{Q^{2}}\big[(Y_T-1)^+\big].$$

**Step 5 — a Black–Scholes call with zero interest rate.** $Y_T=Y_0\exp\!\big(-\tfrac12\sigma^2\tau+\sigma\sqrt\tau Z\big)$ under $Q^2$ (a
martingale, so no drift), and $E[(Y_T-1)^+]$ is a Black–Scholes call with strike $1$ and $r=0$:
$$E^{Q^2}\big[(Y_T-1)^+\big]=Y_0N(d_1)-N(d_2).$$
Multiplying by $S^2$ and using $S^2Y_0=S^1$,
$$\boxed{V=S^{1}N(d_1)-S^{2}N(d_2)}.$$
**Checks.** (i) If $\sigma_2=0$ the second stock is a bond-like asset growing at $r$, and $V$ reduces to the Black–Scholes call on
$S^1$ with strike $S^2e^{r\tau}$. (ii) By parity, exchange call $-$ exchange put $=S^1-S^2$. (iii) The price falls as $\rho$
rises (the ratio becomes less volatile), reaching $(S^1-S^2)^+$ when $\sigma=0$ ($\rho=1$, $\sigma_1=\sigma_2$).

In [23]:
import math, random

def exchange_option(S1, S2, sigma1, sigma2, rho, tau):
    """Margrabe: price of max(S1_T - S2_T, 0).  Independent of the interest rate."""
    sig = math.sqrt(sigma1 ** 2 + sigma2 ** 2 - 2 * rho * sigma1 * sigma2)
    if sig * math.sqrt(tau) < 1e-12:
        return max(S1 - S2, 0.0)
    d1 = (math.log(S1 / S2) + 0.5 * sig ** 2 * tau) / (sig * math.sqrt(tau)); d2 = d1 - sig * math.sqrt(tau)
    return S1 * N(d1) - S2 * N(d2)

def exchange_put(S1, S2, sigma1, sigma2, rho, tau):
    """max(S2_T - S1_T, 0) = call with the roles swapped."""
    return exchange_option(S2, S1, sigma2, sigma1, rho, tau)

S1, S2, s1, s2, rho, r, tau = 110.0, 100.0, 0.30, 0.20, 0.40, 0.05, 1.0
sig = math.sqrt(s1 ** 2 + s2 ** 2 - 2 * rho * s1 * s2)
margrabe = exchange_option(S1, S2, s1, s2, rho, tau)
print(f"ratio volatility sigma = sqrt(s1^2 + s2^2 - 2 rho s1 s2) = {sig:.5f}")
print(f"Margrabe price = {margrabe:.5f}")

# (1) direct risk-neutral Monte Carlo of the two correlated stocks
rng = random.Random(5); n = 300000; tot = 0.0
for _ in range(n):
    z1 = rng.gauss(0, 1); z2 = rho * z1 + math.sqrt(1 - rho ** 2) * rng.gauss(0, 1)
    a = S1 * math.exp((r - s1 ** 2 / 2) * tau + s1 * math.sqrt(tau) * z1)
    b = S2 * math.exp((r - s2 ** 2 / 2) * tau + s2 * math.sqrt(tau) * z2)
    tot += max(a - b, 0.0)
print(f"direct Monte Carlo under Q            = {math.exp(-r * tau) * tot / n:.5f}")

# (2) Solution 2: simulate the ratio Y = S1/S2 as a DRIFTLESS GBM under the S2-numeraire measure, price = S2 * E[(Y_T - 1)^+]
rng = random.Random(6); tot = 0.0; Y0 = S1 / S2
for _ in range(n):
    Y = Y0 * math.exp(-0.5 * sig ** 2 * tau + sig * math.sqrt(tau) * rng.gauss(0, 1))
    tot += max(Y - 1.0, 0.0)
print(f"numeraire method: S2 * E^(Q2)[(Y_T-1)+] = {S2 * tot / n:.5f}")

print(f"\nthe price does not depend on r: r=0 -> {math.exp(0) * margrabe:.5f} (same formula, no r)")
print(f"sigma2 -> 0: Margrabe {exchange_option(110, 100, 0.3, 1e-9, 0.0, tau):.5f}  vs  Black-Scholes call with K = S2 e^(r tau): "
      f"{bs(110, 100 * math.exp(r * tau), r, 0.3, tau):.5f}")
print(f"parity: call - put = {margrabe - exchange_put(S1, S2, s1, s2, rho, tau):.5f}  vs  S1 - S2 = {S1 - S2:.5f}")
print("effect of correlation:  rho =", "  ".join(f"{p:+.1f}: {exchange_option(S1, S2, s1, s2, p, tau):.3f}" for p in (-0.8, 0.0, 0.4, 0.8, 1.0)))

ratio volatility sigma = sqrt(s1^2 + s2^2 - 2 rho s1 s2) = 0.28636
Margrabe price = 17.60533
direct Monte Carlo under Q            = 17.64307
numeraire method: S2 * E^(Q2)[(Y_T-1)+] = 17.61387

the price does not depend on r: r=0 -> 17.60533 (same formula, no r)
sigma2 -> 0: Margrabe 18.14101  vs  Black-Scholes call with K = S2 e^(r tau): 18.14101
parity: call - put = 10.00000  vs  S1 - S2 = 10.00000
effect of correlation:  rho = -0.8: 25.115  +0.0: 20.538  +0.4: 17.605  +0.8: 13.717  +1.0: 10.954


---
*More of Chapter 6 as I keep reading.*